# 03 — mT-MDS Similarity Network

**Run top-to-bottom.** This notebook reads the processed bank-quarter panel saved by Notebook 02. It does not rely on variables from another Jupyter kernel. The TIE calculation is chronologically sorted before YTD→quarterly conversion.

## 1. Imports and load saved data

In [14]:
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx

PROJECT_ROOT = Path("/Users/irfanhussain/Desktop/systemic-risk-project")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

bank_panel_path = PROCESSED_DIR / "bank_quarter_panel.parquet"
ml_path = PROCESSED_DIR / "bank_failure_early_warning.parquet"

assert bank_panel_path.exists(), (
    "Run 02_failure_labels_FIXED.ipynb first so bank_quarter_panel.parquet exists."
)

bank_quarter_panel = pd.read_parquet(bank_panel_path)
ml_data = pd.read_parquet(ml_path) if ml_path.exists() else None

print("Bank panel:", bank_quarter_panel.shape)


Bank panel: (157921, 25)


## 2. Build paper-aligned TIE correctly

In [15]:
def ytd_to_quarterly_simple(df, col):
    df = df.sort_values(["IDRSSD", "Reporting Period End Date"]).copy()
    df["year"] = df["Reporting Period End Date"].dt.year
    df["quarter"] = df["Reporting Period End Date"].dt.quarter
    prev = df.groupby(["IDRSSD", "year"])[col].shift(1)
    prev_q = df.groupby(["IDRSSD", "year"])["quarter"].shift(1)
    out = np.where(
        df["quarter"] == 1,
        df[col],
        np.where(prev_q == df["quarter"] - 1, df[col] - prev, np.nan)
    )
    return pd.Series(out, index=df.index)


FFIEC_DIR = PROJECT_ROOT / "data" / "raw" / "ffiec"
years = sorted(int(p.name) for p in FFIEC_DIR.iterdir() if p.is_dir() and p.name.isdigit())

tie_frames = []

for year in years:
    path = FFIEC_DIR / str(year) / "part2.txt"
    df = pd.read_csv(
        path, sep="\t", dtype=str,
        usecols=["IDRSSD", "Reporting Period End Date", "RIAD4073", "RIAD4107"]
    )
    if len(df) > 0 and pd.isna(df.iloc[0]["IDRSSD"]):
        df = df.iloc[1:].copy()

    df["IDRSSD"] = df["IDRSSD"].astype("string").str.strip()
    df["Reporting Period End Date"] = pd.to_datetime(df["Reporting Period End Date"], errors="coerce")
    df["ytd_interest_expense"] = pd.to_numeric(df["RIAD4073"], errors="coerce")
    df["ytd_interest_income"] = pd.to_numeric(df["RIAD4107"], errors="coerce")

    # Critical: sort first.
    df = df.sort_values(["IDRSSD", "Reporting Period End Date"]).reset_index(drop=True)

    df["quarterly_interest_expense"] = ytd_to_quarterly_simple(df, "ytd_interest_expense")
    df["quarterly_interest_income"] = ytd_to_quarterly_simple(df, "ytd_interest_income")
    df["tie_ratio"] = np.where(
        df["quarterly_interest_income"] != 0,
        df["quarterly_interest_expense"] / df["quarterly_interest_income"],
        np.nan
    )
    df["tie_ratio"] = df["tie_ratio"].replace([np.inf, -np.inf], np.nan)

    tie_frames.append(df[["IDRSSD", "Reporting Period End Date", "quarterly_interest_expense", "quarterly_interest_income", "tie_ratio"]])

tie_panel = pd.concat(tie_frames, ignore_index=True)

# Replace any old TIE columns cleanly.
bank_quarter_panel = bank_quarter_panel.drop(
    columns=[c for c in ["quarterly_interest_expense", "quarterly_interest_income", "tie_ratio"] if c in bank_quarter_panel.columns]
)

bank_quarter_panel = bank_quarter_panel.merge(
    tie_panel,
    on=["IDRSSD", "Reporting Period End Date"],
    how="left",
    validate="one_to_one"
)

print("Valid TIE observations:", bank_quarter_panel["tie_ratio"].notna().sum())


Valid TIE observations: 157801


## 3. Select the 300-bank network snapshot

In [16]:
if ml_data is not None:
    network_date = ml_data["Reporting Period End Date"].max()
else:
    network_date = bank_quarter_panel["Reporting Period End Date"].max()

print("Network date:", network_date)

snapshot = bank_quarter_panel[
    bank_quarter_panel["Reporting Period End Date"] == network_date
].dropna(subset=["IDRSSD", "total_assets"]).copy()

snapshot = snapshot.sort_values("total_assets", ascending=False).head(300).copy()
network_banks = snapshot["IDRSSD"].astype(str).unique()

print("Selected banks:", len(network_banks))


Network date: 2025-06-30 00:00:00
Selected banks: 300


## 4. Build the 20-quarter history

In [17]:
lookback_quarters = 20
all_dates = bank_quarter_panel["Reporting Period End Date"].dropna().drop_duplicates().sort_values()
history_dates = all_dates[all_dates <= network_date].tail(lookback_quarters)

network_history = bank_quarter_panel[
    bank_quarter_panel["IDRSSD"].astype(str).isin(network_banks)
    & bank_quarter_panel["Reporting Period End Date"].isin(history_dates)
].copy()

network_history["IDRSSD"] = network_history["IDRSSD"].astype(str)
print("History quarters:", len(history_dates))
print("History rows:", len(network_history))


History quarters: 20
History rows: 6000


## 5. Build the two paper-aligned correlation layers

In [18]:
network_features = ["capital_ratio", "tie_ratio"]
threshold = 0.8
layer_correlations = {}
layer_edges = {}

for feature in network_features:
    print("Building layer:", feature)

    pivot = network_history.pivot(
        index="Reporting Period End Date",
        columns="IDRSSD",
        values=feature
    ).reindex(columns=network_banks)

    corr = pivot.corr(method="pearson", min_periods=10)
    layer_correlations[feature] = corr

    edge_matrix = corr >= threshold
    np.fill_diagonal(edge_matrix.values, False)
    rows, cols = np.where(np.triu(edge_matrix.values, k=1))

    edges = {(corr.index[i], corr.columns[j]) for i, j in zip(rows, cols)}
    layer_edges[feature] = edges

    print("  Nodes:", len(corr))
    print("  Edges:", len(edges))

final_edges = layer_edges["capital_ratio"] & layer_edges["tie_ratio"]
print("Final mT-MDS edges:", len(final_edges))


Building layer: capital_ratio
  Nodes: 300
  Edges: 4851
Building layer: tie_ratio
  Nodes: 300
  Edges: 41687
Final mT-MDS edges: 4523


## 6. Build the final similarity graph

In [19]:
G_mtds = nx.Graph()
G_mtds.add_nodes_from(network_banks)
G_mtds.add_edges_from(final_edges)

isolated_nodes = list(nx.isolates(G_mtds))
components = list(nx.connected_components(G_mtds))
component_sizes = sorted([len(c) for c in components], reverse=True)

print("Nodes:", G_mtds.number_of_nodes())
print("Edges:", G_mtds.number_of_edges())
print("Isolated:", len(isolated_nodes))
print("Interconnected:", G_mtds.number_of_nodes() - len(isolated_nodes))
print("Components:", len(components))
print("Largest component:", component_sizes[0] if component_sizes else 0)


Nodes: 300
Edges: 4523
Isolated: 57
Interconnected: 243
Components: 59
Largest component: 241


## 7. Exact Minimum Dominating Set (MILP)

In [20]:
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import lil_matrix


def exact_mds_component(G_component):
    nodes = list(G_component.nodes())
    n = len(nodes)
    node_to_idx = {node: i for i, node in enumerate(nodes)}

    A = lil_matrix((n, n), dtype=float)
    for node in nodes:
        i = node_to_idx[node]
        A[i, i] = 1.0
        for neighbor in G_component.neighbors(node):
            A[i, node_to_idx[neighbor]] = 1.0

    result = milp(
        c=np.ones(n),
        integrality=np.ones(n),
        bounds=Bounds(np.zeros(n), np.ones(n)),
        constraints=LinearConstraint(A.tocsr(), np.ones(n), np.full(n, np.inf)),
        options={"presolve": True}
    )

    if not result.success:
        raise RuntimeError(result.message)

    selected = {nodes[i] for i, value in enumerate(result.x) if value > 0.5}
    return selected, result


exact_mds = set()
for component_nodes in sorted(nx.connected_components(G_mtds), key=len, reverse=True):
    H = G_mtds.subgraph(component_nodes).copy()
    selected, result = exact_mds_component(H)
    exact_mds.update(selected)

print("Exact MDS size:", len(exact_mds))
print("Isolated nodes:", len(isolated_nodes))


Exact MDS size: 88
Isolated nodes: 57


## 8. Verify domination

In [21]:
uncovered = []
for node in G_mtds.nodes():
    if node in exact_mds:
        continue
    if not any(neighbor in exact_mds for neighbor in G_mtds.neighbors(node)):
        uncovered.append(node)

print("Uncovered nodes:", len(uncovered))
print("All nodes dominated:", len(uncovered) == 0)
print("Isolated nodes in MDS:", len(set(isolated_nodes) & exact_mds))


Uncovered nodes: 0
All nodes dominated: True
Isolated nodes in MDS: 57


## 9. Exact MDS representative-bank table

In [22]:
degree_dict = dict(G_mtds.degree())

exact_mds_banks = snapshot[
    snapshot["IDRSSD"].astype(str).isin(exact_mds)
].copy()

exact_mds_banks["degree"] = (
    exact_mds_banks["IDRSSD"].astype(str).map(degree_dict)
)

exact_mds_banks = exact_mds_banks.sort_values("degree", ascending=False)

display(
    exact_mds_banks[[
        "IDRSSD", "Financial Institution Name",
        "total_assets", "capital_ratio", "degree"
    ]]
)


,IDRSSD,Financial Institution Name,total_assets,capital_ratio,degree
19656,202907,"COMMUNITY BANK, NATIONAL ASSOCIATION",16405831,0.090943,73
131549,816603,"WASHINGTON TRUST COMPANY, OF WESTERLY, THE",6744900,0.080696,67
48594,3150447,"CHARLES SCHWAB BANK, SSB",247664000,0.085749,54
101108,601416,FIVE STAR BANK,6105894,0.102658,51
70279,3938186,"BANKUNITED, NATIONAL ASSOCIATION",35441030,0.088488,50
...,...,...,...,...,...
83100,480228,"BANK OF AMERICA, NATIONAL ASSOCIATION",2665555000,0.093992,0
17991,1917301,"LAKE FOREST BANK & TRUST COMPANY, NATIONAL ASS...",9544049,0.088332,0
7173,1225761,WELLS FARGO NATIONAL BANK WEST,9406702,0.164003,0
73045,412751,WOODFOREST NATIONAL BANK,9113582,0.077809,0


## 10. Assign every bank to an MDS representative

In [23]:
assignments = []
for node in G_mtds.nodes():
    if node in exact_mds:
        representative = node
    else:
        reps = [n for n in G_mtds.neighbors(node) if n in exact_mds]
        representative = max(reps, key=lambda n: G_mtds.degree(n)) if reps else None
    assignments.append({"IDRSSD": node, "MDS_representative": representative})

assignment_df = pd.DataFrame(assignments)
print("Assigned:", assignment_df["MDS_representative"].notna().sum())
print("Unassigned:", assignment_df["MDS_representative"].isna().sum())

exact_neighborhood_sizes = (
    assignment_df
    .groupby("MDS_representative")
    .size()
    .reset_index(name="neighborhood_size")
    .sort_values("neighborhood_size", ascending=False)
)

exact_neighborhood_sizes = exact_neighborhood_sizes.merge(
    snapshot[["IDRSSD", "Financial Institution Name", "total_assets"]],
    left_on="MDS_representative", right_on="IDRSSD", how="left"
)

display(exact_neighborhood_sizes.head(20))


Assigned: 300
Unassigned: 0


,MDS_representative,neighborhood_size,IDRSSD,Financial Institution Name,total_assets
0,202907,72,202907,"COMMUNITY BANK, NATIONAL ASSOCIATION",16405831
1,3150447,39,3150447,"CHARLES SCHWAB BANK, SSB",247664000
2,100562,19,100562,TRI COUNTIES BANK,9922103
3,677176,17,677176,TRUSTCO BANK,6347651
4,816603,15,816603,"WASHINGTON TRUST COMPANY, OF WESTERLY, THE",6744900
5,352772,7,352772,BANNER BANK,16431475
6,601416,6,601416,FIVE STAR BANK,6105894
7,633378,6,633378,KEARNY BANK,7727514
8,3783313,6,3783313,CROSS RIVER BANK,7936820
9,2925666,5,2925666,PINNACLE BANK,54690073


In [27]:
rcri_path = FULL_DIR / "FFIEC CDR Call Schedule RCRI 06302025.txt"

print("Using file:")
print(rcri_path)

print("\nExists:", rcri_path.exists())

rcri = pd.read_csv(
    rcri_path,
    sep="\t",
    dtype=str,
    low_memory=False,
    nrows=5
)

print("\nShape of first 5 rows:", rcri.shape)

Using file:
/Users/irfanhussain/Desktop/systemic-risk-project/data/raw/ffiec_full_test/FFIEC CDR Call Schedule RCRI 06302025.txt

Exists: True

Shape of first 5 rows: (5, 155)


In [28]:
print("\nFields containing 8274:")
print([
    c for c in rcri.columns
    if "8274" in str(c)
])


Fields containing 8274:
['RCFA8274', 'RCOA8274']


In [29]:
print("\nAll columns:")
print(rcri.columns.tolist())


All columns:
['IDRSSD', 'RCFA2170', 'RCFA3128', 'RCFA3792', 'RCFA5310', 'RCFA5311', 'RCFA7204', 'RCFA7205', 'RCFA7206', 'RCFA8274', 'RCFAA223', 'RCFAA224', 'RCFAB530', 'RCFAB596', 'RCFAH015', 'RCFAH036', 'RCFAH311', 'RCFAH313', 'RCFAH314', 'RCFAJJ30', 'RCFAJJ31', 'RCFAJJ32', 'RCFAKW00', 'RCFAKW03', 'RCFAKX77', 'RCFAKX78', 'RCFAKX79', 'RCFAKX80', 'RCFAKX81', 'RCFAKX82', 'RCFAKX83', 'RCFALB58', 'RCFALB59', 'RCFALB60', 'RCFALB61', 'RCFAP742', 'RCFAP793', 'RCFAP839', 'RCFAP840', 'RCFAP841', 'RCFAP842', 'RCFAP843', 'RCFAP844', 'RCFAP846', 'RCFAP847', 'RCFAP848', 'RCFAP849', 'RCFAP850', 'RCFAP852', 'RCFAP857', 'RCFAP858', 'RCFAP859', 'RCFAP860', 'RCFAP861', 'RCFAP862', 'RCFAP863', 'RCFAP864', 'RCFAP865', 'RCFAP866', 'RCFAP867', 'RCFAP868', 'RCFAP870', 'RCFAP872', 'RCFAP875', 'RCFAQ258', 'RCFAS540', 'RCFW3792', 'RCFW5310', 'RCFW5311', 'RCFW7205', 'RCFW7206', 'RCFWA223', 'RCFWH312', 'RCFWP793', 'RCFWP851', 'RCFWP852', 'RCFWP853', 'RCFWP854', 'RCFWP855', 'RCFWP856', 'RCFWP857', 'RCFWP858', 'RC

In [30]:
# Check the two Tier-1 capital fields

print("RCFA8274 exists:", "RCFA8274" in rcri.columns)
print("RCOA8274 exists:", "RCOA8274" in rcri.columns)

display(
    rcri[
        [
            "IDRSSD",
            "RCFA8274",
            "RCOA8274"
        ]
    ].head(20)
)

RCFA8274 exists: True
RCOA8274 exists: True


,IDRSSD,RCFA8274,RCOA8274
0,NaN,TIER 1 CPTL ALLWBL UNDR RISK-BASED,TIER 1 CPTL ALLWBL UNDR RISK-BASED
1,37,NaN,17403
2,242,NaN,6112
3,279,NaN,49742
4,354,NaN,8859


In [31]:
# Convert the two fields to numeric

rcri["RCFA8274_num"] = pd.to_numeric(
    rcri["RCFA8274"],
    errors="coerce"
)

rcri["RCOA8274_num"] = pd.to_numeric(
    rcri["RCOA8274"],
    errors="coerce"
)

print("RCFA8274 non-null:",
      rcri["RCFA8274_num"].notna().sum())

print("RCOA8274 non-null:",
      rcri["RCOA8274_num"].notna().sum())

print("Both non-null:",
      (
          rcri["RCFA8274_num"].notna()
          & rcri["RCOA8274_num"].notna()
      ).sum())

RCFA8274 non-null: 0
RCOA8274 non-null: 4
Both non-null: 0


In [32]:
rcri["tier1_capital"] = (
    rcri["RCFA8274_num"]
    .combine_first(rcri["RCOA8274_num"])
)

print(
    "Tier-1 capital available:",
    rcri["tier1_capital"].notna().sum()
)

display(
    rcri[
        [
            "IDRSSD",
            "RCFA8274_num",
            "RCOA8274_num",
            "tier1_capital"
        ]
    ].head(20)
)

Tier-1 capital available: 4


,IDRSSD,RCFA8274_num,RCOA8274_num,tier1_capital
0,NaN,NaN,NaN,NaN
1,37,NaN,17403.0,17403.0
2,242,NaN,6112.0,6112.0
3,279,NaN,49742.0,49742.0
4,354,NaN,8859.0,8859.0


In [33]:
rcri["RCFA2170_num"] = pd.to_numeric(
    rcri["RCFA2170"],
    errors="coerce"
)

rcri["RCOA2170_num"] = pd.to_numeric(
    rcri["RCOA2170"],
    errors="coerce"
)

rcri["total_assets"] = (
    rcri["RCFA2170_num"]
    .combine_first(rcri["RCOA2170_num"])
)

rcri["t1crc"] = (
    rcri["tier1_capital"]
    / rcri["total_assets"]
)

display(
    rcri[
        [
            "IDRSSD",
            "tier1_capital",
            "total_assets",
            "t1crc"
        ]
    ].head(20)
)

,IDRSSD,tier1_capital,total_assets,t1crc
0,NaN,NaN,NaN,NaN
1,37,17403.0,82892.0,0.209948
2,242,6112.0,56675.0,0.107843
3,279,49742.0,443000.0,0.112284
4,354,8859.0,101574.0,0.087217


In [35]:
# Clean the RCRI 2025-06-30 data

rcri_clean = rcri.copy()

# Remove the metadata/header row
if len(rcri_clean) > 0 and pd.isna(rcri_clean.iloc[0]["IDRSSD"]):
    rcri_clean = rcri_clean.iloc[1:].copy()

# Clean IDRSSD
rcri_clean["IDRSSD"] = (
    rcri_clean["IDRSSD"]
    .astype("string")
    .str.strip()
)

# Convert Tier-1 capital fields to numeric
rcri_clean["RCFA8274"] = pd.to_numeric(
    rcri_clean["RCFA8274"],
    errors="coerce"
)

rcri_clean["RCOA8274"] = pd.to_numeric(
    rcri_clean["RCOA8274"],
    errors="coerce"
)

# Unified Tier-1 capital
rcri_clean["tier1_capital"] = (
    rcri_clean["RCFA8274"]
    .combine_first(rcri_clean["RCOA8274"])
)

print("Rows:", len(rcri_clean))
print("Unique IDRSSD:", rcri_clean["IDRSSD"].nunique())
print("Tier-1 non-null:", rcri_clean["tier1_capital"].notna().sum())

rcri_clean["Reporting Period End Date"] = pd.Timestamp("2025-06-30")

Rows: 4
Unique IDRSSD: 4
Tier-1 non-null: 4


/var/folders/ls/00jcg6g10sq2mljk4_p4j52w0000gn/T/ipykernel_6477/495598710.py:30: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  .combine_first(rcri_clean["RCOA8274"])


In [36]:
panel_snapshot = bank_quarter_panel[
    bank_quarter_panel["Reporting Period End Date"]
    == pd.Timestamp("2025-06-30")
].copy()

print("Bank panel rows:", len(panel_snapshot))
print("RCRI rows:", len(rcri_clean))

print(
    "Panel unique IDRSSD:",
    panel_snapshot["IDRSSD"].astype("string").nunique()
)

print(
    "RCRI unique IDRSSD:",
    rcri_clean["IDRSSD"].nunique()
)

Bank panel rows: 4477
RCRI rows: 4
Panel unique IDRSSD: 4477
RCRI unique IDRSSD: 4


In [37]:
panel_ids = set(
    panel_snapshot["IDRSSD"]
    .astype("string")
    .dropna()
)

rcri_ids = set(
    rcri_clean["IDRSSD"]
    .dropna()
)

print("Panel IDs:", len(panel_ids))
print("RCRI IDs:", len(rcri_ids))
print("Common IDs:", len(panel_ids & rcri_ids))
print("Only in panel:", len(panel_ids - rcri_ids))
print("Only in RCRI:", len(rcri_ids - panel_ids))

Panel IDs: 4477
RCRI IDs: 4
Common IDs: 4
Only in panel: 4473
Only in RCRI: 0


In [38]:
tier1_2025 = rcri_clean[
    [
        "IDRSSD",
        "Reporting Period End Date",
        "tier1_capital"
    ]
].copy()

merged_test = panel_snapshot.merge(
    tier1_2025,
    on=["IDRSSD", "Reporting Period End Date"],
    how="left",
    validate="one_to_one"
)

print("Rows after merge:", len(merged_test))

print(
    "Tier-1 matched:",
    merged_test["tier1_capital"].notna().sum()
)

print(
    "Tier-1 missing:",
    merged_test["tier1_capital"].isna().sum()
)

Rows after merge: 4477
Tier-1 matched: 4
Tier-1 missing: 4473


In [39]:
merged_test["t1crc"] = (
    merged_test["tier1_capital"]
    / merged_test["total_assets"]
)

merged_test["t1crc"] = merged_test["t1crc"].replace(
    [np.inf, -np.inf],
    np.nan
)

display(
    merged_test[
        [
            "IDRSSD",
            "Financial Institution Name",
            "total_assets",
            "tier1_capital",
            "t1crc"
        ]
    ].head(20)
)

,IDRSSD,Financial Institution Name,total_assets,tier1_capital,t1crc
0,1000052,WASHINGTON COUNTY BANK,615402,NaN,<NA>
1,1000276,PITTSFIELD CO-OPERATIVE BANK,422901,NaN,<NA>
2,1000641,CITIZENS STATE BANK OF NEW CASTLE,852045,NaN,<NA>
3,1000856,"BAXTER STATE BANK, THE",24007,NaN,<NA>
4,1000959,EAGLE BANK,215926,NaN,<NA>
5,1001059,SECURITY BANK & TRUST COMPANY,1214409,NaN,<NA>
6,1001152,INTERNATIONAL BANK OF COMMERCE,9726584,NaN,<NA>
7,1001639,PARKWAY BANK AND TRUST COMPANY,3548084,NaN,<NA>
8,1001648,CARROLLTON BANK,3686498,NaN,<NA>
9,1001853,CITIZENS BANK,352129,NaN,<NA>


In [40]:
rcri_path = (
    FULL_DIR /
    "FFIEC CDR Call Schedule RCRI 06302025.txt"
)

rcri_full = pd.read_csv(
    rcri_path,
    sep="\t",
    dtype=str,
    usecols=[
        "IDRSSD",
        "RCFA8274",
        "RCOA8274"
    ]
)

print("Rows loaded:", len(rcri_full))
print("Columns:", rcri_full.columns.tolist())

Rows loaded: 4478
Columns: ['IDRSSD', 'RCFA8274', 'RCOA8274']


In [42]:
rcri_full = rcri_full.copy()

if len(rcri_full) > 0 and pd.isna(rcri_full.iloc[0]["IDRSSD"]):
    rcri_full = rcri_full.iloc[1:].copy()

rcri_full["IDRSSD"] = (
    rcri_full["IDRSSD"]
    .astype("string")
    .str.strip()
)

rcri_full["RCFA8274"] = pd.to_numeric(
    rcri_full["RCFA8274"],
    errors="coerce"
)

rcri_full["RCOA8274"] = pd.to_numeric(
    rcri_full["RCOA8274"],
    errors="coerce"
)

rcri_full["tier1_capital"] = (
    rcri_full["RCFA8274"]
    .combine_first(rcri_full["RCOA8274"])
)

print("Rows after cleaning:", len(rcri_full))
print("Unique IDRSSD:", rcri_full["IDRSSD"].nunique())
print(
    "Tier-1 capital available:",
    rcri_full["tier1_capital"].notna().sum()
)

rcri_full["Reporting Period End Date"] = pd.Timestamp("2025-06-30")

Rows after cleaning: 4477
Unique IDRSSD: 4477
Tier-1 capital available: 4477


In [43]:
panel_snapshot = bank_quarter_panel[
    bank_quarter_panel["Reporting Period End Date"]
    == pd.Timestamp("2025-06-30")
].copy()

panel_ids = set(
    panel_snapshot["IDRSSD"]
    .astype("string")
    .dropna()
)

rcri_ids = set(
    rcri_full["IDRSSD"]
    .dropna()
)

print("Panel unique IDs:", len(panel_ids))
print("RCRI unique IDs:", len(rcri_ids))
print("Common IDs:", len(panel_ids & rcri_ids))
print("Only in panel:", len(panel_ids - rcri_ids))
print("Only in RCRI:", len(rcri_ids - panel_ids))

Panel unique IDs: 4477
RCRI unique IDs: 4477
Common IDs: 4477
Only in panel: 0
Only in RCRI: 0


In [44]:
tier1_2025 = rcri_full[
    [
        "IDRSSD",
        "Reporting Period End Date",
        "tier1_capital"
    ]
].copy()

merged_test = panel_snapshot.merge(
    tier1_2025,
    on=[
        "IDRSSD",
        "Reporting Period End Date"
    ],
    how="left",
    validate="one_to_one"
)

print("Panel rows:", len(panel_snapshot))
print("Merged rows:", len(merged_test))
print(
    "Tier-1 matched:",
    merged_test["tier1_capital"].notna().sum()
)
print(
    "Tier-1 missing:",
    merged_test["tier1_capital"].isna().sum()
)

Panel rows: 4477
Merged rows: 4477
Tier-1 matched: 4477
Tier-1 missing: 0


In [45]:
merged_test["t1crc"] = (
    merged_test["tier1_capital"]
    / merged_test["total_assets"]
)

merged_test["t1crc"] = merged_test["t1crc"].replace(
    [np.inf, -np.inf],
    np.nan
)

display(
    merged_test[
        [
            "IDRSSD",
            "Financial Institution Name",
            "total_assets",
            "tier1_capital",
            "t1crc"
        ]
    ].head(20)
)

,IDRSSD,Financial Institution Name,total_assets,tier1_capital,t1crc
0,1000052,WASHINGTON COUNTY BANK,615402,76699.0,0.124632
1,1000276,PITTSFIELD CO-OPERATIVE BANK,422901,61818.0,0.146176
2,1000641,CITIZENS STATE BANK OF NEW CASTLE,852045,88230.0,0.103551
3,1000856,"BAXTER STATE BANK, THE",24007,5300.0,0.220769
4,1000959,EAGLE BANK,215926,30469.0,0.141109
5,1001059,SECURITY BANK & TRUST COMPANY,1214409,108919.0,0.089689
6,1001152,INTERNATIONAL BANK OF COMMERCE,9726584,1694427.0,0.174206
7,1001639,PARKWAY BANK AND TRUST COMPANY,3548084,389561.0,0.109795
8,1001648,CARROLLTON BANK,3686498,295442.0,0.080142
9,1001853,CITIZENS BANK,352129,44035.0,0.125054


In [46]:
# Sanity check for T1CRC on 2025-06-30

print("Total bank observations:", len(merged_test))

print(
    "Valid T1CRC:",
    merged_test["t1crc"].notna().sum()
)

print(
    "Missing T1CRC:",
    merged_test["t1crc"].isna().sum()
)

print("\nT1CRC summary:")
print(
    merged_test["t1crc"]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
)

print("\nNegative T1CRC:")
print((merged_test["t1crc"] < 0).sum())

print("\nT1CRC > 1:")
print((merged_test["t1crc"] > 1).sum())

Total bank observations: 4477
Valid T1CRC: 4477
Missing T1CRC: 0

T1CRC summary:
count      4477.0
mean     0.130948
std      0.105209
min      0.032504
1%       0.072969
5%       0.081939
25%      0.094746
50%      0.108139
75%        0.1299
95%      0.211415
99%      0.846452
max      1.096982
Name: t1crc, dtype: Float64

Negative T1CRC:
0

T1CRC > 1:
1


In [47]:
# Compare our old proxy capital ratio with the new T1CRC

comparison_check = merged_test[
    [
        "IDRSSD",
        "Financial Institution Name",
        "capital_ratio",
        "t1crc"
    ]
].dropna()

print(
    "Correlation between old proxy and T1CRC:",
    comparison_check["capital_ratio"].corr(comparison_check["t1crc"])
)

display(comparison_check.head(20))

Correlation between old proxy and T1CRC: 0.9715296685106594


,IDRSSD,Financial Institution Name,capital_ratio,t1crc
0,1000052,WASHINGTON COUNTY BANK,0.116293,0.124632
1,1000276,PITTSFIELD CO-OPERATIVE BANK,0.130094,0.146176
2,1000641,CITIZENS STATE BANK OF NEW CASTLE,0.100401,0.103551
3,1000856,"BAXTER STATE BANK, THE",0.244845,0.220769
4,1000959,EAGLE BANK,0.135495,0.141109
5,1001059,SECURITY BANK & TRUST COMPANY,0.094905,0.089689
6,1001152,INTERNATIONAL BANK OF COMMERCE,0.186796,0.174206
7,1001639,PARKWAY BANK AND TRUST COMPANY,0.111524,0.109795
8,1001648,CARROLLTON BANK,0.078018,0.080142
9,1001853,CITIZENS BANK,0.124809,0.125054


In [48]:
from pathlib import Path
import pandas as pd
import numpy as np

RCRI_DIR = Path("/Users/irfanhussain/Desktop/systemic-risk-project/data/raw/ffiec/rcri")

# --------------------------------------------------
# 1. Find all RCRI files
# --------------------------------------------------
rcri_files = sorted(RCRI_DIR.glob("FFIEC CDR Call Schedule RCRI *.txt"))

print("RCRI files found:", len(rcri_files))

for f in rcri_files:
    print(f.name)

RCRI files found: 20
FFIEC CDR Call Schedule RCRI 03312021.txt
FFIEC CDR Call Schedule RCRI 03312022.txt
FFIEC CDR Call Schedule RCRI 03312023.txt
FFIEC CDR Call Schedule RCRI 03312024.txt
FFIEC CDR Call Schedule RCRI 03312025.txt
FFIEC CDR Call Schedule RCRI 06302021.txt
FFIEC CDR Call Schedule RCRI 06302022.txt
FFIEC CDR Call Schedule RCRI 06302023.txt
FFIEC CDR Call Schedule RCRI 06302024.txt
FFIEC CDR Call Schedule RCRI 06302025.txt
FFIEC CDR Call Schedule RCRI 09302020.txt
FFIEC CDR Call Schedule RCRI 09302021.txt
FFIEC CDR Call Schedule RCRI 09302022.txt
FFIEC CDR Call Schedule RCRI 09302023.txt
FFIEC CDR Call Schedule RCRI 09302024.txt
FFIEC CDR Call Schedule RCRI 12312020.txt
FFIEC CDR Call Schedule RCRI 12312021.txt
FFIEC CDR Call Schedule RCRI 12312022.txt
FFIEC CDR Call Schedule RCRI 12312023.txt
FFIEC CDR Call Schedule RCRI 12312024.txt


In [49]:
# --------------------------------------------------
# 2. Read Tier-1 capital from every quarter
# --------------------------------------------------

rcri_list = []

for path in rcri_files:

    df = pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        usecols=["IDRSSD", "RCFA8274", "RCOA8274"]
    )

    # Remove metadata/header row if present
    if len(df) > 0 and pd.isna(df.iloc[0]["IDRSSD"]):
        df = df.iloc[1:].copy()

    df["IDRSSD"] = (
        df["IDRSSD"]
        .astype("string")
        .str.strip()
    )

    df["RCFA8274"] = pd.to_numeric(
        df["RCFA8274"], errors="coerce"
    )

    df["RCOA8274"] = pd.to_numeric(
        df["RCOA8274"], errors="coerce"
    )

    # Use whichever reporting-basis field is populated
    df["tier1_capital"] = (
        df["RCFA8274"]
        .combine_first(df["RCOA8274"])
    )

    # Extract date from filename: MMDDYYYY
    date_str = path.stem.split()[-1]
    report_date = pd.to_datetime(date_str, format="%m%d%Y")

    df["Reporting Period End Date"] = report_date

    rcri_list.append(
        df[
            [
                "IDRSSD",
                "Reporting Period End Date",
                "tier1_capital"
            ]
        ]
    )

tier1_history = pd.concat(rcri_list, ignore_index=True)

print("Shape:", tier1_history.shape)
print("Quarters:", tier1_history["Reporting Period End Date"].nunique())
print("Banks:", tier1_history["IDRSSD"].nunique())

Shape: (95282, 3)
Quarters: 20
Banks: 5139


In [50]:
tier1_history.groupby(
    "Reporting Period End Date"
).agg(
    banks=("IDRSSD", "nunique"),
    non_missing_tier1=("tier1_capital", "count")
).sort_index()

,banks,non_missing_tier1
Reporting Period End Date,,
2020-09-30,5082,5082
2020-12-31,5050,5050
2021-03-31,5027,5027
2021-06-30,4998,4998
2021-09-30,4962,4962
2021-12-31,4887,4887
2022-03-31,4844,4844
2022-06-30,4821,4821
2022-09-30,4796,4796


In [51]:
# --------------------------------------------------
# Merge historical Tier-1 capital into bank panel
# --------------------------------------------------

# Make sure keys have the same type
bank_quarter_panel["IDRSSD"] = (
    bank_quarter_panel["IDRSSD"]
    .astype("string")
    .str.strip()
)

tier1_history["IDRSSD"] = (
    tier1_history["IDRSSD"]
    .astype("string")
    .str.strip()
)

# Check uniqueness before merging
print("Duplicate Tier-1 keys:",
      tier1_history.duplicated(
          ["IDRSSD", "Reporting Period End Date"]
      ).sum())

# Merge
bank_panel_t1 = bank_quarter_panel.merge(
    tier1_history,
    on=["IDRSSD", "Reporting Period End Date"],
    how="left",
    validate="one_to_one"
)

# Calculate exact T1 capital ratio
bank_panel_t1["t1crc"] = (
    bank_panel_t1["tier1_capital"]
    / bank_panel_t1["total_assets"]
)

# Remove invalid ratios
bank_panel_t1["t1crc"] = bank_panel_t1["t1crc"].replace(
    [np.inf, -np.inf],
    np.nan
)

print("Original panel shape:", bank_quarter_panel.shape)
print("New panel shape:", bank_panel_t1.shape)

print("\nT1CRC missing:",
      bank_panel_t1["t1crc"].isna().sum())

print("T1CRC available:",
      bank_panel_t1["t1crc"].notna().sum())

Duplicate Tier-1 keys: 0
Original panel shape: (157921, 28)
New panel shape: (157921, 30)

T1CRC missing: 62639
T1CRC available: 95282


In [52]:
# Coverage by quarter

coverage = (
    bank_panel_t1
    .groupby("Reporting Period End Date")
    .agg(
        banks=("IDRSSD", "nunique"),
        t1crc_available=("t1crc", "count")
    )
    .sort_index()
)

coverage["coverage_pct"] = (
    100 * coverage["t1crc_available"] / coverage["banks"]
)

coverage

,banks,t1crc_available,coverage_pct
Reporting Period End Date,,,
2018-03-31,5657,0,0.0
2018-06-30,5593,0,0.0
2018-09-30,5527,0,0.0
2018-12-31,5456,0,0.0
2019-03-31,5411,0,0.0
2019-06-30,5352,0,0.0
2019-09-30,5308,0,0.0
2019-12-31,5227,0,0.0
2020-03-31,5165,0,0.0


In [53]:
bank_panel_t1["t1crc"].describe()

count     95282.0
mean     0.124101
std      0.100804
min      0.000122
25%      0.090265
50%      0.102543
75%       0.12303
max      1.096982
Name: t1crc, dtype: Float64

In [54]:
bank_panel_t1[
    ["IDRSSD", "Reporting Period End Date",
     "total_assets", "tier1_capital", "t1crc"]
].head(20)

,IDRSSD,Reporting Period End Date,total_assets,tier1_capital,t1crc
0,1000052,2018-03-31,360585,NaN,<NA>
1,1000052,2018-06-30,365600,NaN,<NA>
2,1000052,2018-09-30,375344,NaN,<NA>
3,1000052,2018-12-31,432110,NaN,<NA>
4,1000052,2019-03-31,416494,NaN,<NA>
5,1000052,2019-06-30,429465,NaN,<NA>
6,1000052,2019-09-30,433506,NaN,<NA>
7,1000052,2019-12-31,441212,NaN,<NA>
8,1000052,2020-03-31,424954,NaN,<NA>
9,1000052,2020-06-30,468217,NaN,<NA>


In [55]:
# --------------------------------------------------
# Check the exact 20-quarter T1CRC window
# --------------------------------------------------

NETWORK_START = pd.Timestamp("2020-09-30")
NETWORK_END   = pd.Timestamp("2025-06-30")

network_panel = bank_panel_t1[
    (bank_panel_t1["Reporting Period End Date"] >= NETWORK_START) &
    (bank_panel_t1["Reporting Period End Date"] <= NETWORK_END)
].copy()

print("Network panel shape:", network_panel.shape)
print(
    "Number of quarters:",
    network_panel["Reporting Period End Date"].nunique()
)

# T1CRC coverage inside the network window
print(
    "\nT1CRC missing inside network window:",
    network_panel["t1crc"].isna().sum()
)

print("\nT1CRC summary:")
print(network_panel["t1crc"].describe())

Network panel shape: (95282, 30)
Number of quarters: 20

T1CRC missing inside network window: 0

T1CRC summary:
count     95282.0
mean     0.124101
std      0.100804
min      0.000122
25%      0.090265
50%      0.102543
75%       0.12303
max      1.096982
Name: t1crc, dtype: Float64


In [56]:
# --------------------------------------------------
# Find banks having complete T1CRC history
# --------------------------------------------------

quarter_count = (
    network_panel
    .groupby("IDRSSD")["Reporting Period End Date"]
    .nunique()
)

t1_complete = (
    network_panel
    .groupby("IDRSSD")["t1crc"]
    .count()
)

complete_banks = quarter_count[
    (quarter_count == 20) &
    (t1_complete == 20)
].index

print("Banks with complete 20-quarter T1CRC history:",
      len(complete_banks))

Banks with complete 20-quarter T1CRC history: 4421


In [57]:
# --------------------------------------------------
# Select 300 largest banks with complete T1CRC history
# --------------------------------------------------

END_DATE = pd.Timestamp("2025-06-30")
N_BANKS = 300

# Snapshot at final quarter
snapshot = network_panel[
    (network_panel["Reporting Period End Date"] == END_DATE) &
    (network_panel["IDRSSD"].isin(complete_banks))
].copy()

# Rank by total assets
top_banks = (
    snapshot
    .sort_values("total_assets", ascending=False)
    .head(N_BANKS)
    .copy()
)

selected_banks = top_banks["IDRSSD"].tolist()

print("Selected banks:", len(selected_banks))

print("\nAsset range:")
print("Largest:", top_banks["total_assets"].max())
print("Smallest:", top_banks["total_assets"].min())

print("\nTop 10:")
print(
    top_banks[
        ["IDRSSD", "total_assets", "t1crc"]
    ].head(10)
)

Selected banks: 300

Asset range:
Largest: 3788551000
Smallest: 4603395

Top 10:
         IDRSSD  total_assets     t1crc
136739   852218    3788551000  0.074517
83100    480228    2665555000  0.073616
82468    476810    1833933000  0.087084
78737    451965    1746394000  0.085748
86750    504713     671438131  0.090439
5050     112837     648909392  0.104532
22676   2182786     625410000   0.10279
131645   817824     554573142  0.090931
136804   852320     535585000  0.094792
92216    541101     398293000  0.060009


In [58]:
# --------------------------------------------------
# Build bank × quarter T1CRC matrix
# --------------------------------------------------

t1_matrix = (
    network_panel[
        network_panel["IDRSSD"].isin(selected_banks)
    ]
    .pivot(
        index="IDRSSD",
        columns="Reporting Period End Date",
        values="t1crc"
    )
    .sort_index(axis=1)
)

print("Matrix shape:", t1_matrix.shape)
print("Missing values:", t1_matrix.isna().sum().sum())

t1_matrix.head()

Matrix shape: (300, 20)
Missing values: 0


Reporting Period End Date,2020-09-30,2020-12-31,2021-03-31,2021-06-30,2021-09-30,2021-12-31,2022-03-31,2022-06-30,2022-09-30,2022-12-31,2023-03-31,2023-06-30,2023-09-30,2023-12-31,2024-03-31,2024-06-30,2024-09-30,2024-12-31,2025-03-31,2025-06-30
IDRSSD,,,,,,,,,,,,,,,,,,,,
1001152,0.127886,0.123175,0.123848,0.12438,0.1253,0.138583,0.136719,0.133259,0.136915,0.149808,0.142707,0.153705,0.155101,0.160899,0.160043,0.175601,0.164591,0.177009,0.172995,0.174206
1002878,0.094165,0.097002,0.09307,0.096906,0.099164,0.100462,0.092453,0.096432,0.100863,0.102223,0.094214,0.096198,0.096244,0.095958,0.09572,0.097325,0.09215,0.092735,0.094823,0.0957
100562,0.094748,0.093694,0.092716,0.093893,0.093402,0.09429,0.093329,0.094018,0.097836,0.101704,0.104665,0.106308,0.108178,0.109889,0.112201,0.11399,0.115296,0.11873,0.118747,0.117693
1011526,0.090771,0.084794,0.084518,0.086235,0.084952,0.082729,0.086265,0.084493,0.090982,0.087204,0.087969,0.094353,0.097238,0.089386,0.092228,0.096601,0.100069,0.087494,0.090087,0.095818
1017939,0.091048,0.091482,0.086735,0.088991,0.090362,0.083194,0.087682,0.090322,0.094525,0.09415,0.092222,0.092109,0.091712,0.089421,0.091679,0.09373,0.092556,0.092417,0.090425,0.091617


In [59]:
# --------------------------------------------------
# T1CRC correlation matrix
# --------------------------------------------------

corr_matrix = t1_matrix.T.corr(method="pearson")

print("Correlation matrix shape:", corr_matrix.shape)

print("\nCorrelation statistics:")
print(
    corr_matrix.values[
        np.triu_indices_from(corr_matrix.values, k=1)
    ].min(),
    corr_matrix.values[
        np.triu_indices_from(corr_matrix.values, k=1)
    ].max()
)

Correlation matrix shape: (300, 300)

Correlation statistics:
-0.9869171110029329 0.992063909386421


In [60]:
# --------------------------------------------------
# Create T1CRC network layer at threshold 0.8
# --------------------------------------------------

THRESHOLD = 0.80

adjacency_t1 = (
    corr_matrix >= THRESHOLD
)

# Remove self-links
np.fill_diagonal(
    adjacency_t1.values,
    False
)

# Number of undirected edges
num_edges_t1 = np.triu(
    adjacency_t1.values,
    k=1
).sum()

num_nodes_t1 = len(selected_banks)

print("Nodes:", num_nodes_t1)
print("Edges:", int(num_edges_t1))
print(
    "Density:",
    num_edges_t1 / (num_nodes_t1 * (num_nodes_t1 - 1) / 2)
)

Nodes: 300
Edges: 4185
Density: 0.09331103678929765


In [61]:
# --------------------------------------------------
# Check TIE coverage for our 300 selected banks
# --------------------------------------------------

tie_panel = network_panel[
    network_panel["IDRSSD"].isin(selected_banks)
].copy()

print("Rows:", len(tie_panel))
print("Banks:", tie_panel["IDRSSD"].nunique())
print(
    "Quarters:",
    tie_panel["Reporting Period End Date"].nunique()
)

print("\nTIE missing:")
print(tie_panel["tie_ratio"].isna().sum())

print("\nTIE summary:")
print(tie_panel["tie_ratio"].describe())

Rows: 6000
Banks: 300
Quarters: 20

TIE missing:
0

TIE summary:
count    6000.000000
mean        0.251396
std         0.195484
min        -0.300546
25%         0.072096
50%         0.227170
75%         0.401532
max         1.279055
Name: tie_ratio, dtype: float64


In [62]:
# --------------------------------------------------
# Build bank × quarter TIE matrix
# --------------------------------------------------

tie_matrix = (
    tie_panel
    .pivot(
        index="IDRSSD",
        columns="Reporting Period End Date",
        values="tie_ratio"
    )
    .sort_index(axis=1)
)

print("TIE matrix shape:", tie_matrix.shape)
print("Missing values:", tie_matrix.isna().sum().sum())

tie_matrix.head()

TIE matrix shape: (300, 20)
Missing values: 0


Reporting Period End Date,2020-09-30,2020-12-31,2021-03-31,2021-06-30,2021-09-30,2021-12-31,2022-03-31,2022-06-30,2022-09-30,2022-12-31,2023-03-31,2023-06-30,2023-09-30,2023-12-31,2024-03-31,2024-06-30,2024-09-30,2024-12-31,2025-03-31,2025-06-30
IDRSSD,,,,,,,,,,,,,,,,,,,,
1001152,0.077942,0.074173,0.070536,0.068988,0.067209,0.067594,0.075112,0.100452,0.132465,0.155883,0.175072,0.204449,0.210525,0.230772,0.238466,0.256489,0.257075,0.256257,0.247394,0.245578
1002878,0.072877,0.057654,0.048625,0.044633,0.041738,0.040642,0.038221,0.031108,0.027005,0.058902,0.137519,0.213662,0.253398,0.292061,0.321623,0.327909,0.319786,0.302155,0.272515,0.282506
100562,0.021639,0.016348,0.013855,0.012179,0.012352,0.009917,0.009890,0.009810,0.010346,0.025012,0.077283,0.157349,0.200080,0.237253,0.267786,0.283156,0.279800,0.264922,0.261523,0.241758
1011526,0.121883,0.101888,0.083146,0.063828,0.049592,0.043393,0.041294,0.034020,0.039777,0.079749,0.129734,0.178625,0.206687,0.238887,0.260513,0.281348,0.287375,0.295030,0.278945,0.266249
1017939,0.066789,0.043831,0.039550,0.027181,0.022786,0.020280,0.018544,0.023080,0.047244,0.112150,0.232136,0.313079,0.364707,0.391782,0.400122,0.390326,0.393038,0.383181,0.362172,0.346375


In [63]:
# --------------------------------------------------
# TIE correlation matrix
# --------------------------------------------------

tie_corr = tie_matrix.T.corr(method="pearson")

print("Correlation matrix shape:", tie_corr.shape)

upper = tie_corr.values[
    np.triu_indices_from(tie_corr.values, k=1)
]

print("Minimum correlation:", upper.min())
print("Maximum correlation:", upper.max())

Correlation matrix shape: (300, 300)
Minimum correlation: -0.8653558397255854
Maximum correlation: 0.9992361790905787


In [64]:
# --------------------------------------------------
# TIE network layer
# --------------------------------------------------

THRESHOLD = 0.80

adjacency_tie = tie_corr >= THRESHOLD

# Remove self-loops
np.fill_diagonal(adjacency_tie.values, False)

num_edges_tie = np.triu(
    adjacency_tie.values,
    k=1
).sum()

num_nodes_tie = len(tie_matrix)

print("Nodes:", num_nodes_tie)
print("Edges:", int(num_edges_tie))

print(
    "Density:",
    num_edges_tie /
    (num_nodes_tie * (num_nodes_tie - 1) / 2)
)

Nodes: 300
Edges: 41687
Density: 0.9294760312151616


In [65]:
# --------------------------------------------------
# Multivariate network = T1CRC AND TIE
# --------------------------------------------------

adjacency_multi = (
    adjacency_t1.values &
    adjacency_tie.values
)

# Remove self-loops
np.fill_diagonal(adjacency_multi, False)

num_edges_multi = np.triu(
    adjacency_multi,
    k=1
).sum()

num_nodes_multi = adjacency_multi.shape[0]

possible_edges = (
    num_nodes_multi * (num_nodes_multi - 1) // 2
)

print("Multivariate network")
print("--------------------")
print("Nodes:", num_nodes_multi)
print("Edges:", int(num_edges_multi))
print("Possible edges:", possible_edges)
print(
    "Density:",
    num_edges_multi / possible_edges
)

Multivariate network
--------------------
Nodes: 300
Edges: 3905
Possible edges: 44850
Density: 0.0870680044593088


In [66]:
import networkx as nx

# Create graph
G_multi = nx.from_numpy_array(adjacency_multi)

# Attach IDRSSD to nodes
id_map = {
    i: bank_id
    for i, bank_id in enumerate(t1_matrix.index)
}

G_multi = nx.relabel_nodes(G_multi, id_map)

print("Nodes:", G_multi.number_of_nodes())
print("Edges:", G_multi.number_of_edges())

isolated = list(nx.isolates(G_multi))

print("Isolated nodes:", len(isolated))

components = list(nx.connected_components(G_multi))

print("Connected components:", len(components))

largest_component = max(
    components,
    key=len
)

print(
    "Largest component size:",
    len(largest_component)
)

Nodes: 300
Edges: 3905
Isolated nodes: 84
Connected components: 90
Largest component size: 182


In [67]:
# --------------------------------------------------
# Verify multivariate edges are a subset of both layers
# --------------------------------------------------

t1_edges = np.triu(adjacency_t1.values, k=1)
tie_edges = np.triu(adjacency_tie.values, k=1)
multi_edges = np.triu(adjacency_multi, k=1)

print("T1CRC edges:", int(t1_edges.sum()))
print("TIE edges:", int(tie_edges.sum()))
print("Multi-layer edges:", int(multi_edges.sum()))

print(
    "Multi ⊆ T1CRC:",
    np.all(multi_edges <= t1_edges)
)

print(
    "Multi ⊆ TIE:",
    np.all(multi_edges <= tie_edges)
)

T1CRC edges: 4185
TIE edges: 41687
Multi-layer edges: 3905
Multi ⊆ T1CRC: True
Multi ⊆ TIE: True


In [68]:
# --------------------------------------------------
# Exact Minimum Dominating Set (MDS)
# --------------------------------------------------

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.optimize import milp, LinearConstraint, Bounds

mds_nodes = []

components = list(nx.connected_components(G_multi))

print("Solving", len(components), "connected components...")

for component_id, component in enumerate(components):

    nodes = list(component)
    n = len(nodes)

    # Local node index
    local_index = {
        node: i for i, node in enumerate(nodes)
    }

    # Build domination constraint matrix
    rows = []
    cols = []
    data = []

    for node in nodes:

        i = local_index[node]

        # Node itself must count
        rows.append(i)
        cols.append(i)
        data.append(1)

        # All neighbors count
        for neighbor in G_multi.neighbors(node):

            if neighbor in local_index:
                j = local_index[neighbor]

                rows.append(i)
                cols.append(j)
                data.append(1)

    A = sp.coo_matrix(
        (data, (rows, cols)),
        shape=(n, n)
    ).tocsr()

    # Objective: minimize number of selected nodes
    c = np.ones(n)

    # x_i ∈ {0,1}
    integrality = np.ones(n)

    bounds = Bounds(
        np.zeros(n),
        np.ones(n)
    )

    # Domination constraint:
    # x_i + sum(neighbors) >= 1
    constraint = LinearConstraint(
        A,
        np.ones(n),
        np.full(n, np.inf)
    )

    result = milp(
        c=c,
        integrality=integrality,
        bounds=bounds,
        constraints=constraint,
        options={"time_limit": 60}
    )

    if not result.success:
        print(
            f"Component {component_id}: "
            f"MILP failed -> {result.message}"
        )
        continue

    selected_local = np.where(result.x > 0.5)[0]

    selected_global = [
        nodes[i]
        for i in selected_local
    ]

    mds_nodes.extend(selected_global)

print("\nExact MDS completed.")
print("MDS size:", len(mds_nodes))
print("MDS fraction:", len(mds_nodes) / G_multi.number_of_nodes())

Solving 90 connected components...

Exact MDS completed.
MDS size: 117
MDS fraction: 0.39


In [69]:
# --------------------------------------------------
# Verify domination
# --------------------------------------------------

mds_set = set(mds_nodes)

uncovered = []

for node in G_multi.nodes():

    dominated = (
        node in mds_set or
        any(neighbor in mds_set
            for neighbor in G_multi.neighbors(node))
    )

    if not dominated:
        uncovered.append(node)

print("MDS nodes:", len(mds_set))
print("Uncovered nodes:", len(uncovered))

if len(uncovered) == 0:
    print("✓ Every bank is dominated.")
else:
    print("✗ Some banks are not dominated.")
    print(uncovered[:20])

MDS nodes: 117
Uncovered nodes: 0
✓ Every bank is dominated.


In [70]:
# --------------------------------------------------
# Isolated nodes must belong to MDS
# --------------------------------------------------

isolated_nodes = set(nx.isolates(G_multi))

isolated_in_mds = isolated_nodes.intersection(mds_set)

print("Isolated nodes:", len(isolated_nodes))
print("Isolated nodes in MDS:", len(isolated_in_mds))

print(
    "All isolated nodes selected:",
    isolated_nodes == isolated_in_mds
)

Isolated nodes: 84
Isolated nodes in MDS: 84
All isolated nodes selected: True


In [71]:
# --------------------------------------------------
# MDS bank information
# --------------------------------------------------

mds_table = top_banks[
    top_banks["IDRSSD"].isin(mds_set)
].copy()

mds_table["degree"] = mds_table["IDRSSD"].map(
    dict(G_multi.degree())
)

mds_table["is_isolated"] = (
    mds_table["degree"] == 0
)

mds_table = mds_table.sort_values(
    ["degree", "total_assets"],
    ascending=False
)

print("MDS size:", len(mds_table))

mds_table[
    [
        "IDRSSD",
        "total_assets",
        "t1crc",
        "degree",
        "is_isolated"
    ]
].head(20)

MDS size: 117


,IDRSSD,total_assets,t1crc,degree,is_isolated
16564,1842065,60355912,0.130276,92,False
118337,723112,209311000,0.101667,88,False
48594,3150447,247664000,0.129668,87,False
113102,682563,51489892,0.088104,73,False
49977,3212149,114294933,0.090138,58,False
99025,587800,8473696,0.131845,57,False
25805,2354985,22541660,0.088252,36,False
86750,504713,671438131,0.090439,32,False
130596,811046,7542729,0.09805,21,False
121878,749242,26605052,0.093242,20,False


In [72]:
# --------------------------------------------------
# Find bank-identification columns
# --------------------------------------------------

[c for c in bank_panel_t1.columns
 if any(word in c.upper() for word in ["NAME", "CERT", "RSSD"])]

['IDRSSD', 'FDIC Certificate Number', 'Financial Institution Name']

In [73]:
# --------------------------------------------------
# Add bank names to MDS results
# --------------------------------------------------

mds_table = top_banks[
    top_banks["IDRSSD"].isin(mds_set)
].copy()

# Add graph degree
degree_map = dict(G_multi.degree())

mds_table["degree"] = (
    mds_table["IDRSSD"].map(degree_map)
)

# Mark isolated banks
mds_table["is_isolated"] = (
    mds_table["degree"] == 0
)

# Sort by degree first, then assets
mds_table = mds_table.sort_values(
    ["degree", "total_assets"],
    ascending=False
)

print("MDS size:", len(mds_table))

mds_table[
    [
        "Financial Institution Name",
        "IDRSSD",
        "total_assets",
        "t1crc",
        "degree",
        "is_isolated"
    ]
].head(30)

MDS size: 117


,Financial Institution Name,IDRSSD,total_assets,t1crc,degree,is_isolated
16564,CIBC BANK USA,1842065,60355912,0.130276,92,False
118337,"FIFTH THIRD BANK, NATIONAL ASSOCIATION",723112,209311000,0.101667,88,False
48594,"CHARLES SCHWAB BANK, SSB",3150447,247664000,0.129668,87,False
113102,FROST BANK,682563,51489892,0.088104,73,False
49977,UBS BANK USA,3212149,114294933,0.090138,58,False
99025,LIBERTY BANK,587800,8473696,0.131845,57,False
25805,CUSTOMERS BANK,2354985,22541660,0.088252,36,False
86750,U.S. BANK NATIONAL ASSOCIATION,504713,671438131,0.090439,32,False
130596,BANKERS TRUST COMPANY,811046,7542729,0.09805,21,False
121878,RENASANT BANK,749242,26605052,0.093242,20,False


In [74]:
# --------------------------------------------------
# Compare MDS banks with the 300-bank universe
# --------------------------------------------------

print("Average assets - all 300:",
      top_banks["total_assets"].mean())

print("Average assets - MDS:",
      mds_table["total_assets"].mean())

print("\nMedian assets - all 300:",
      top_banks["total_assets"].median())

print("Median assets - MDS:",
      mds_table["total_assets"].median())

print("\nAverage degree - all 300:",
      pd.Series(dict(G_multi.degree())).mean())

print("Average degree - MDS:",
      mds_table["degree"].mean())

Average assets - all 300: 74691412.45
Average assets - MDS: 111148206.35897435

Median assets - all 300: 10668597.5
Median assets - MDS: 12205300.0

Average degree - all 300: 26.033333333333335
Average degree - MDS: 6.042735042735043


In [75]:
# --------------------------------------------------
# How many MDS banks are also among the largest?
# --------------------------------------------------

for k in [10, 25, 50, 100]:
    
    largest_k = set(
        top_banks
        .nlargest(k, "total_assets")["IDRSSD"]
    )
    
    overlap = len(mds_set.intersection(largest_k))
    
    print(
        f"Top {k} assets ∩ MDS:",
        overlap
    )

Top 10 assets ∩ MDS: 6
Top 25 assets ∩ MDS: 14
Top 50 assets ∩ MDS: 23
Top 100 assets ∩ MDS: 43


In [76]:
# --------------------------------------------------
# Assign every bank to an MDS representative
# --------------------------------------------------

mds_set = set(mds_nodes)

degree_map = dict(G_multi.degree())

# Asset lookup
asset_map = (
    top_banks
    .set_index("IDRSSD")["total_assets"]
    .to_dict()
)

assignments = []

for node in G_multi.nodes():

    # An MDS node represents itself
    if node in mds_set:
        representative = node

    else:
        # MDS neighbors of this node
        mds_neighbors = [
            nbr
            for nbr in G_multi.neighbors(node)
            if nbr in mds_set
        ]

        if len(mds_neighbors) == 0:
            representative = None
        else:
            # Choose the most connected MDS neighbor
            representative = max(
                mds_neighbors,
                key=lambda x: (
                    degree_map[x],
                    asset_map.get(x, 0)
                )
            )

    assignments.append(
        {
            "IDRSSD": node,
            "MDS_representative": representative
        }
    )

assignment_df = pd.DataFrame(assignments)

print("Banks:", len(assignment_df))
print(
    "Unassigned:",
    assignment_df["MDS_representative"].isna().sum()
)

Banks: 300
Unassigned: 0


In [77]:
# --------------------------------------------------
# Coverage count per MDS representative
# --------------------------------------------------

coverage_count = (
    assignment_df
    .dropna(subset=["MDS_representative"])
    .groupby("MDS_representative")
    .size()
    .rename("banks_represented")
)

mds_ranked = mds_table.copy()

mds_ranked["banks_represented"] = (
    mds_ranked["IDRSSD"]
    .map(coverage_count)
    .fillna(0)
    .astype(int)
)

mds_ranked = mds_ranked.sort_values(
    ["banks_represented", "degree", "total_assets"],
    ascending=False
)

mds_ranked[
    [
        "Financial Institution Name",
        "IDRSSD",
        "total_assets",
        "t1crc",
        "degree",
        "banks_represented",
        "is_isolated"
    ]
].head(30)

,Financial Institution Name,IDRSSD,total_assets,t1crc,degree,banks_represented,is_isolated
16564,CIBC BANK USA,1842065,60355912,0.130276,92,88,False
118337,"FIFTH THIRD BANK, NATIONAL ASSOCIATION",723112,209311000,0.101667,88,14,False
145473,THIRD FEDERAL SAVINGS & LOAN ASSOCIATION OF CL...,915878,17363064,0.099887,12,12,False
130596,BANKERS TRUST COMPANY,811046,7542729,0.09805,21,9,False
25805,CUSTOMERS BANK,2354985,22541660,0.088252,36,7,False
99025,LIBERTY BANK,587800,8473696,0.131845,57,6,False
48594,"CHARLES SCHWAB BANK, SSB",3150447,247664000,0.129668,87,5,False
57797,TRUSTMARK NATIONAL BANK,342634,18613278,0.100164,12,5,False
131123,CITY NATIONAL BANK OF FLORIDA,814430,27059543,0.109335,19,4,False
32575,PRINCIPAL BANK,2654911,8964155,0.076473,13,4,False


In [78]:
# --------------------------------------------------
# MDS composition
# --------------------------------------------------

print("Total MDS:", len(mds_set))
print("Forced isolated MDS:", len(isolated_nodes))
print(
    "Non-isolated MDS:",
    len(mds_set - isolated_nodes)
)

print(
    "Percentage of MDS forced by isolation:",
    100 * len(isolated_nodes) / len(mds_set)
)

Total MDS: 117
Forced isolated MDS: 84
Non-isolated MDS: 33
Percentage of MDS forced by isolation: 71.7948717948718


In [79]:
# --------------------------------------------------
# Threshold sensitivity analysis
# --------------------------------------------------

from scipy.optimize import milp, LinearConstraint, Bounds
import scipy.sparse as sp

thresholds = [0.70, 0.75, 0.80, 0.85, 0.90]

sensitivity_results = []


def solve_exact_mds(G):
    """
    Solve exact Minimum Dominating Set component-by-component.
    Returns selected node set.
    """

    selected = []

    for component in nx.connected_components(G):

        nodes = list(component)
        n = len(nodes)

        # Isolated node
        if n == 1:
            selected.append(nodes[0])
            continue

        local_index = {
            node: i for i, node in enumerate(nodes)
        }

        rows = []
        cols = []
        data = []

        for node in nodes:

            i = local_index[node]

            # The node itself
            rows.append(i)
            cols.append(i)
            data.append(1)

            # Its neighbors
            for nbr in G.neighbors(node):

                j = local_index[nbr]

                rows.append(i)
                cols.append(j)
                data.append(1)

        A = sp.coo_matrix(
            (data, (rows, cols)),
            shape=(n, n)
        ).tocsr()

        result = milp(
            c=np.ones(n),
            integrality=np.ones(n),
            bounds=Bounds(
                np.zeros(n),
                np.ones(n)
            ),
            constraints=LinearConstraint(
                A,
                np.ones(n),
                np.full(n, np.inf)
            )
        )

        if not result.success:
            raise RuntimeError(
                f"MDS optimization failed: {result.message}"
            )

        selected.extend(
            nodes[i]
            for i in np.where(result.x > 0.5)[0]
        )

    return set(selected)


for threshold in thresholds:

    # T1CRC layer
    t1_adj = corr_matrix >= threshold
    np.fill_diagonal(t1_adj.values, False)

    # TIE layer
    tie_adj = tie_corr >= threshold
    np.fill_diagonal(tie_adj.values, False)

    # Multivariate intersection
    multi_adj = (
        t1_adj.values &
        tie_adj.values
    )

    np.fill_diagonal(multi_adj, False)

    # Graph
    G = nx.from_numpy_array(multi_adj)

    # Relabel nodes with IDRSSD
    G = nx.relabel_nodes(
        G,
        {i: bank_id for i, bank_id in enumerate(t1_matrix.index)}
    )

    # Statistics
    nodes = G.number_of_nodes()
    edges = G.number_of_edges()
    possible = nodes * (nodes - 1) // 2

    isolated_count = len(list(nx.isolates(G)))

    components = list(nx.connected_components(G))
    num_components = len(components)

    largest_component = max(
        len(c) for c in components
    )

    # Exact MDS
    mds = solve_exact_mds(G)

    sensitivity_results.append({
        "threshold": threshold,
        "edges": edges,
        "density": edges / possible,
        "isolated_nodes": isolated_count,
        "components": num_components,
        "largest_component": largest_component,
        "mds_size": len(mds),
        "mds_fraction": len(mds) / nodes
    })


sensitivity_df = pd.DataFrame(
    sensitivity_results
)

sensitivity_df

,threshold,edges,density,isolated_nodes,components,largest_component,mds_size,mds_fraction
0,0.70,7238,0.161382,39,42,257,71,0.236667
1,0.75,5545,0.123634,59,64,200,89,0.296667
2,0.80,3905,0.087068,84,90,182,117,0.390000
3,0.85,2389,0.053266,120,125,153,152,0.506667
4,0.90,1069,0.023835,160,172,109,192,0.640000


In [80]:
# --------------------------------------------------
# Save the current paper-aligned network results
# --------------------------------------------------

from pathlib import Path

PROCESSED_DIR = Path(
    "/Users/irfanhussain/Desktop/systemic-risk-project/data/processed"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Save selected 300-bank universe
top_banks.to_parquet(
    PROCESSED_DIR / "mtds_300_bank_universe.parquet",
    index=False
)

# Save T1CRC and TIE matrices
t1_matrix.to_parquet(
    PROCESSED_DIR / "mtds_t1crc_matrix.parquet"
)

tie_matrix.to_parquet(
    PROCESSED_DIR / "mtds_tie_matrix.parquet"
)

# Save correlation matrices
corr_matrix.to_parquet(
    PROCESSED_DIR / "mtds_t1crc_correlation.parquet"
)

tie_corr.to_parquet(
    PROCESSED_DIR / "mtds_tie_correlation.parquet"
)

# Save sensitivity results
sensitivity_df.to_csv(
    PROCESSED_DIR / "mtds_threshold_sensitivity.csv",
    index=False
)

# Save MDS result
mds_ranked.to_parquet(
    PROCESSED_DIR / "mtds_representatives.parquet",
    index=False
)

print("Saved successfully.")

Saved successfully.


In [81]:
# --------------------------------------------------
# Save primary multivariate network (rho = 0.80)
# --------------------------------------------------

multi_edges = []

for i, bank_i in enumerate(t1_matrix.index):

    for j, bank_j in enumerate(t1_matrix.index):

        if i < j and adjacency_multi[i, j]:

            multi_edges.append({
                "IDRSSD_1": bank_i,
                "IDRSSD_2": bank_j
            })

multi_edges_df = pd.DataFrame(multi_edges)

multi_edges_df.to_csv(
    PROCESSED_DIR / "mtds_multivariate_edges_rho_080.csv",
    index=False
)

print("Edges saved:", len(multi_edges_df))

Edges saved: 3905


In [82]:
# --------------------------------------------------
# Load failure early-warning data
# --------------------------------------------------

failure_path = PROCESSED_DIR / "bank_failure_early_warning.parquet"

ew = pd.read_parquet(failure_path)

print("Shape:", ew.shape)
print("\nColumns:")
print(ew.columns.tolist())

Shape: (149092, 30)

Columns:
['IDRSSD', 'FDIC Certificate Number', 'Reporting Period End Date', 'Financial Institution Name', 'Financial Institution State', 'Financial Institution Filing Type', 'total_assets', 'total_equity', 'loans_net_unearned', 'total_nonaccrual', 'total_deposits', 'ytd_net_interest_income', 'ytd_net_income', 'year', 'quarter', 'quarterly_net_interest_income', 'quarterly_net_income', 'capital_ratio', 'loan_to_asset', 'deposit_to_asset', 'nonaccrual_to_loan', 'nonaccrual_to_asset', 'roa', 'nii_to_asset', 'has_nonaccrual', 'failure_date', 'prediction_end', 'complete_4q_horizon', 'failure_next_4q', 'after_failure']


In [83]:
# --------------------------------------------------
# Failure data for the 300-bank MDS universe
# --------------------------------------------------

ew_300 = ew[
    (ew["IDRSSD"].isin(selected_banks)) &
    (ew["Reporting Period End Date"] >= NETWORK_START) &
    (ew["Reporting Period End Date"] <= NETWORK_END)
].copy()

print("Rows:", len(ew_300))
print("Banks:", ew_300["IDRSSD"].nunique())
print(
    "Quarters:",
    ew_300["Reporting Period End Date"].nunique()
)

print("\nTarget distribution:")
print(
    ew_300["failure_next_4q"]
    .value_counts()
    .sort_index()
)

Rows: 6000
Banks: 300
Quarters: 20

Target distribution:
failure_next_4q
0    6000
Name: count, dtype: int64


In [85]:
# --------------------------------------------------
# Cleanly attach bank name + MDS status
# --------------------------------------------------

# Remove any previously created name columns
name_cols = [
    c for c in ew_300.columns
    if "Financial Institution Name" in c
]

ew_300 = ew_300.drop(
    columns=name_cols,
    errors="ignore"
)

# Bank name lookup
name_map = (
    top_banks
    .set_index("IDRSSD")["Financial Institution Name"]
    .to_dict()
)

# Attach name
ew_300["Financial Institution Name"] = (
    ew_300["IDRSSD"].map(name_map)
)

# Attach MDS status
ew_300["is_mds"] = (
    ew_300["IDRSSD"].isin(mds_set)
)

print(
    ew_300[
        [
            "Financial Institution Name",
            "IDRSSD",
            "Reporting Period End Date",
            "failure_next_4q",
            "is_mds"
        ]
    ].head(20)
)

        Financial Institution Name   IDRSSD Reporting Period End Date  \
0   INTERNATIONAL BANK OF COMMERCE  1001152                2020-09-30   
1   INTERNATIONAL BANK OF COMMERCE  1001152                2020-12-31   
2   INTERNATIONAL BANK OF COMMERCE  1001152                2021-03-31   
3   INTERNATIONAL BANK OF COMMERCE  1001152                2021-06-30   
4   INTERNATIONAL BANK OF COMMERCE  1001152                2021-09-30   
5   INTERNATIONAL BANK OF COMMERCE  1001152                2021-12-31   
6   INTERNATIONAL BANK OF COMMERCE  1001152                2022-03-31   
7   INTERNATIONAL BANK OF COMMERCE  1001152                2022-06-30   
8   INTERNATIONAL BANK OF COMMERCE  1001152                2022-09-30   
9   INTERNATIONAL BANK OF COMMERCE  1001152                2022-12-31   
10  INTERNATIONAL BANK OF COMMERCE  1001152                2023-03-31   
11  INTERNATIONAL BANK OF COMMERCE  1001152                2023-06-30   
12  INTERNATIONAL BANK OF COMMERCE  1001152        

In [89]:
# --------------------------------------------------
# Rebuild clean 300-bank failure dataset
# --------------------------------------------------

failure_path = PROCESSED_DIR / "bank_failure_early_warning.parquet"

ew = pd.read_parquet(failure_path)

# Keep only our selected 300 banks
ew_300 = ew[
    ew["IDRSSD"].isin(selected_banks)
].copy()

# Restrict to the 20-quarter network window
ew_300 = ew_300[
    (ew_300["Reporting Period End Date"] >= NETWORK_START) &
    (ew_300["Reporting Period End Date"] <= NETWORK_END)
].copy()

# MDS membership
ew_300["is_mds"] = (
    ew_300["IDRSSD"].isin(mds_set)
)

# Bank-name lookup
name_map = (
    top_banks
    .set_index("IDRSSD")["Financial Institution Name"]
    .to_dict()
)

ew_300["Financial Institution Name"] = (
    ew_300["IDRSSD"].map(name_map)
)

# Asset lookup from the 2025 snapshot
asset_map = (
    top_banks
    .set_index("IDRSSD")["total_assets"]
    .to_dict()
)

ew_300["snapshot_total_assets"] = (
    ew_300["IDRSSD"].map(asset_map)
)

# T1CRC lookup from the 2025 snapshot
t1crc_map = (
    top_banks
    .set_index("IDRSSD")["t1crc"]
    .to_dict()
)

ew_300["snapshot_t1crc"] = (
    ew_300["IDRSSD"].map(t1crc_map)
)

print("Shape:", ew_300.shape)
print("Banks:", ew_300["IDRSSD"].nunique())
print(
    "Quarters:",
    ew_300["Reporting Period End Date"].nunique()
)

print("\nTarget count:")
print(
    ew_300["failure_next_4q"]
    .value_counts()
    .sort_index()
)

Shape: (6000, 33)
Banks: 300
Quarters: 20

Target count:
failure_next_4q
0    6000
Name: count, dtype: int64


In [90]:
# --------------------------------------------------
# Check whether any failed-bank observations
# belong to our 300-bank universe
# --------------------------------------------------

print(
    "Positive labels in ALL banks:",
    ew["failure_next_4q"].sum()
)

print(
    "Positive labels in selected 300:",
    ew_300["failure_next_4q"].sum()
)

failed_ids_300 = set(
    ew_300.loc[
        ew_300["failure_next_4q"] == 1,
        "IDRSSD"
    ].dropna()
)

print(
    "Failed-bank IDs in selected 300:",
    len(failed_ids_300)
)

Positive labels in ALL banks: 68
Positive labels in selected 300: 0
Failed-bank IDs in selected 300: 0


In [91]:
# --------------------------------------------------
# Prepare ALL-bank failure modeling dataset
# --------------------------------------------------

failure_path = PROCESSED_DIR / "bank_failure_early_warning.parquet"

ew = pd.read_parquet(failure_path)

ml_features = [
    "capital_ratio",
    "loan_to_asset",
    "deposit_to_asset",
    "nonaccrual_to_loan",
    "nonaccrual_to_asset",
    "roa",
    "nii_to_asset"
]

target = "failure_next_4q"

required_cols = [
    "IDRSSD",
    "Reporting Period End Date",
    target
] + ml_features

missing_cols = [
    c for c in required_cols
    if c not in ew.columns
]

print("Missing required columns:", missing_cols)

model_data = ew[required_cols].copy()

# Keep observations where the target is known
model_data = model_data[
    model_data[target].notna()
].copy()

model_data = model_data.sort_values(
    "Reporting Period End Date"
).reset_index(drop=True)

print("Model data shape:", model_data.shape)
print("Positive labels:", int(model_data[target].sum()))
print("Negative labels:", int((model_data[target] == 0).sum()))

Missing required columns: []
Model data shape: (149092, 10)
Positive labels: 68
Negative labels: 149024


In [92]:
# --------------------------------------------------
# Temporal split
# --------------------------------------------------

TRAIN_END = pd.Timestamp("2023-12-31")
VALIDATION_START = pd.Timestamp("2024-01-01")
VALIDATION_END = pd.Timestamp("2024-12-31")

train = model_data[
    model_data["Reporting Period End Date"] <= TRAIN_END
].copy()

validation = model_data[
    (model_data["Reporting Period End Date"] >= VALIDATION_START) &
    (model_data["Reporting Period End Date"] <= VALIDATION_END)
].copy()

# We will use 2025 as the final prediction period
prediction_2025 = model_data[
    model_data["Reporting Period End Date"] > VALIDATION_END
].copy()

print("Train:", train.shape)
print("Validation:", validation.shape)
print("2025 prediction set:", prediction_2025.shape)

print("\nPositive labels:")
print("Train:", int(train[target].sum()))
print("Validation:", int(validation[target].sum()))
print("2025:", int(prediction_2025[target].sum()))

Train: (121764, 10)
Validation: (18332, 10)
2025 prediction set: (8996, 10)

Positive labels:
Train: 55
Validation: 9
2025: 4


In [93]:
# --------------------------------------------------
# Prepare X / y
# --------------------------------------------------

from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

X_train = train[ml_features]
y_train = train[target].astype(int)

X_val = validation[ml_features]
y_val = validation[target].astype(int)

X_2025 = prediction_2025[ml_features]
y_2025 = prediction_2025[target].astype(int)

# Median imputation learned ONLY from training data
imputer = SimpleImputer(strategy="median")

X_train_imp = imputer.fit_transform(X_train)
X_val_imp = imputer.transform(X_val)
X_2025_imp = imputer.transform(X_2025)

# Class imbalance
positive = y_train.sum()
negative = len(y_train) - positive

scale_pos_weight = negative / positive

print("Training positives:", positive)
print("Training negatives:", negative)
print("scale_pos_weight:", scale_pos_weight)

Training positives: 55
Training negatives: 121709
scale_pos_weight: 2212.890909090909


In [94]:
# --------------------------------------------------
# XGBoost
# --------------------------------------------------

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,
    random_state=42
)

xgb_model.fit(
    X_train_imp,
    y_train
)

# Validation probabilities
val_prob = xgb_model.predict_proba(
    X_val_imp
)[:, 1]

print("Validation ROC-AUC:",
      roc_auc_score(y_val, val_prob))

print("Validation PR-AUC:",
      average_precision_score(y_val, val_prob))

Validation ROC-AUC: 0.7192478184673786
Validation PR-AUC: 0.001102129415207688


In [95]:
# --------------------------------------------------
# 2025 temporal test
# --------------------------------------------------

prob_2025 = xgb_model.predict_proba(
    X_2025_imp
)[:, 1]

print("2025 ROC-AUC:",
      roc_auc_score(y_2025, prob_2025))

print("2025 PR-AUC:",
      average_precision_score(y_2025, prob_2025))

2025 ROC-AUC: 0.9736432384341637
2025 PR-AUC: 0.47130192502532925


In [101]:
# --------------------------------------------------
# Final model for 2025 prediction
# --------------------------------------------------

final_train = model_data[
    model_data["Reporting Period End Date"] <= VALIDATION_END
].copy()

X_final = final_train[ml_features]
y_final = final_train[target].astype(int)

X_target_2025 = prediction_2025[ml_features]

# Fit imputer on all pre-2025 data
final_imputer = SimpleImputer(strategy="median")

X_final_imp = final_imputer.fit_transform(X_final)
X_target_2025_imp = final_imputer.transform(X_target_2025)

positive_final = y_final.sum()
negative_final = len(y_final) - positive_final

final_model = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=negative_final / positive_final,
    random_state=42
)

final_model.fit(
    X_final_imp,
    y_final
)

prediction_2025["failure_probability"] = (
    final_model.predict_proba(
        X_target_2025_imp
    )[:, 1]
)

print(
    prediction_2025["failure_probability"].describe()
)

count    8996.000000
mean        0.024464
std         0.087333
min         0.000013
25%         0.000357
50%         0.001712
75%         0.008750
max         0.994117
Name: failure_probability, dtype: float64


In [102]:
# --------------------------------------------------
# Correct 2025-06-30 snapshot:
# ONLY our 300-bank network universe
# --------------------------------------------------

risk_snapshot = prediction_2025[
    (prediction_2025["Reporting Period End Date"] == END_DATE) &
    (prediction_2025["IDRSSD"].isin(selected_banks))
].copy()

print("Banks in MDS/network universe:", len(risk_snapshot))
print("Expected:", N_BANKS)

Banks in MDS/network universe: 300
Expected: 300


In [103]:
# --------------------------------------------------
# Attach bank + network information
# --------------------------------------------------

name_map = (
    top_banks
    .set_index("IDRSSD")["Financial Institution Name"]
    .to_dict()
)

asset_map = (
    top_banks
    .set_index("IDRSSD")["total_assets"]
    .to_dict()
)

t1crc_map = (
    top_banks
    .set_index("IDRSSD")["t1crc"]
    .to_dict()
)

degree_map = dict(G_multi.degree())

risk_snapshot["Financial Institution Name"] = (
    risk_snapshot["IDRSSD"].map(name_map)
)

risk_snapshot["total_assets"] = (
    risk_snapshot["IDRSSD"].map(asset_map)
)

risk_snapshot["t1crc"] = (
    risk_snapshot["IDRSSD"].map(t1crc_map)
)

risk_snapshot["is_mds"] = (
    risk_snapshot["IDRSSD"].isin(mds_set)
)

risk_snapshot["network_degree"] = (
    risk_snapshot["IDRSSD"].map(degree_map)
)

risk_snapshot["banks_represented"] = (
    risk_snapshot["IDRSSD"]
    .map(coverage_count)
    .fillna(1)
    .astype(int)
)

print(
    "Missing bank names:",
    risk_snapshot["Financial Institution Name"].isna().sum()
)

print(
    "Missing assets:",
    risk_snapshot["total_assets"].isna().sum()
)

print(
    "Missing network degree:",
    risk_snapshot["network_degree"].isna().sum()
)

print(
    "MDS banks:",
    risk_snapshot["is_mds"].sum()
)

Missing bank names: 0
Missing assets: 0
Missing network degree: 0
MDS banks: 117


In [104]:
# --------------------------------------------------
# Top-risk banks within our 300-bank universe
# --------------------------------------------------

risk_ranking = (
    risk_snapshot[
        [
            "Financial Institution Name",
            "IDRSSD",
            "total_assets",
            "t1crc",
            "failure_probability",
            "network_degree",
            "banks_represented",
            "is_mds"
        ]
    ]
    .sort_values(
        "failure_probability",
        ascending=False
    )
    .reset_index(drop=True)
)

risk_ranking.head(25)

,Financial Institution Name,IDRSSD,total_assets,t1crc,failure_probability,network_degree,banks_represented,is_mds
0,"MORGAN STANLEY PRIVATE BANK, NATIONAL ASSOCIATION",2489805,235034000,0.071815,0.772845,0,1,True
1,BANK OF HAWAII,795968,23677044,0.080824,0.508975,38,1,False
2,"NORTHERN TRUST COMPANY, THE",210434,171309876,0.060000,0.487829,1,1,False
3,UBS BANK USA,3212149,114294933,0.090138,0.375422,58,3,True
4,TRISTATE CAPITAL BANK,3475083,21174085,0.076302,0.359613,0,1,True
5,CARTER BANK & TRUST,3476192,4780209,0.093035,0.300421,5,1,False
6,PRINCIPAL BANK,2654911,8964155,0.076473,0.240463,13,4,True
7,WOODFOREST NATIONAL BANK,412751,9113582,0.088460,0.201734,60,1,False
8,FIRSTBANK,288853,26761023,0.089818,0.170207,95,1,False
9,BELL BANK,929352,14066290,0.098682,0.155887,0,1,True


In [105]:
# --------------------------------------------------
# Compare predicted failure probability
# --------------------------------------------------

risk_comparison = (
    risk_snapshot
    .groupby("is_mds")
    .agg(
        banks=("IDRSSD", "nunique"),
        mean_failure_probability=(
            "failure_probability", "mean"
        ),
        median_failure_probability=(
            "failure_probability", "median"
        ),
        max_failure_probability=(
            "failure_probability", "max"
        )
    )
)

risk_comparison

,banks,mean_failure_probability,median_failure_probability,max_failure_probability
is_mds,,,,
False,183,0.016052,0.000647,0.508975
True,117,0.027233,0.001789,0.772845


In [106]:
# --------------------------------------------------
# Probability distribution in our 300 banks
# --------------------------------------------------

risk_snapshot["failure_probability"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

count    300.000000
mean       0.020412
std        0.074369
min        0.000018
50%        0.000941
75%        0.005498
90%        0.039354
95%        0.104544
99%        0.376546
max        0.772845
Name: failure_probability, dtype: float64

In [107]:
# Distribution of predicted risk

print(
    "Risk > 0.90:",
    (risk_snapshot["failure_probability"] > 0.90).sum()
)

print(
    "Risk > 0.50:",
    (risk_snapshot["failure_probability"] > 0.50).sum()
)

print(
    "Risk < 0.10:",
    (risk_snapshot["failure_probability"] < 0.10).sum()
)

Risk > 0.90: 0
Risk > 0.50: 2
Risk < 0.10: 284


In [108]:
# --------------------------------------------------
# Statistical comparison: MDS vs non-MDS
# --------------------------------------------------

from scipy.stats import mannwhitneyu

mds_risk_values = risk_snapshot.loc[
    risk_snapshot["is_mds"],
    "failure_probability"
]

non_mds_risk_values = risk_snapshot.loc[
    ~risk_snapshot["is_mds"],
    "failure_probability"
]

u_stat, p_value = mannwhitneyu(
    mds_risk_values,
    non_mds_risk_values,
    alternative="greater"
)

print("Mann-Whitney U:", u_stat)
print("p-value:", p_value)

Mann-Whitney U: 12246.0
p-value: 0.01780274461400594


In [109]:
# --------------------------------------------------
# Top 20 distress-risk banks
# --------------------------------------------------

top_risk = (
    risk_snapshot[
        [
            "Financial Institution Name",
            "IDRSSD",
            "failure_probability",
            "is_mds",
            "network_degree",
            "banks_represented",
            "total_assets",
            "t1crc"
        ]
    ]
    .sort_values(
        "failure_probability",
        ascending=False
    )
    .reset_index(drop=True)
)

top_risk.head(20)

,Financial Institution Name,IDRSSD,failure_probability,is_mds,network_degree,banks_represented,total_assets,t1crc
0,"MORGAN STANLEY PRIVATE BANK, NATIONAL ASSOCIATION",2489805,0.772845,True,0,1,235034000,0.071815
1,BANK OF HAWAII,795968,0.508975,False,38,1,23677044,0.080824
2,"NORTHERN TRUST COMPANY, THE",210434,0.487829,False,1,1,171309876,0.060000
3,UBS BANK USA,3212149,0.375422,True,58,3,114294933,0.090138
4,TRISTATE CAPITAL BANK,3475083,0.359613,True,0,1,21174085,0.076302
5,CARTER BANK & TRUST,3476192,0.300421,False,5,1,4780209,0.093035
6,PRINCIPAL BANK,2654911,0.240463,True,13,4,8964155,0.076473
7,WOODFOREST NATIONAL BANK,412751,0.201734,False,60,1,9113582,0.088460
8,FIRSTBANK,288853,0.170207,False,95,1,26761023,0.089818
9,BELL BANK,929352,0.155887,True,0,1,14066290,0.098682


In [110]:
# --------------------------------------------------
# First Systemic-Risk Screening Score
# --------------------------------------------------

score_df = risk_snapshot.copy()

# Distress percentile
score_df["distress_percentile"] = (
    score_df["failure_probability"]
    .rank(pct=True)
)

# Network percentile
score_df["network_percentile"] = (
    score_df["network_degree"]
    .rank(pct=True)
)

# Equal-weight screening score
score_df["systemic_risk_score"] = (
    0.5 * score_df["distress_percentile"] +
    0.5 * score_df["network_percentile"]
)

# Sort
systemic_ranking = (
    score_df[
        [
            "Financial Institution Name",
            "IDRSSD",
            "failure_probability",
            "network_degree",
            "banks_represented",
            "total_assets",
            "t1crc",
            "is_mds",
            "distress_percentile",
            "network_percentile",
            "systemic_risk_score"
        ]
    ]
    .sort_values(
        "systemic_risk_score",
        ascending=False
    )
    .reset_index(drop=True)
)

systemic_ranking.head(25)

,Financial Institution Name,IDRSSD,failure_probability,network_degree,banks_represented,total_assets,t1crc,is_mds,distress_percentile,network_percentile,systemic_risk_score
0,FIRSTBANK,288853,0.170207,95,1,26761023,0.089818,False,0.973333,0.998333,0.985833
1,LAKE CITY BANK,874845,0.022619,94,1,6957623,0.123954,False,0.870000,0.993333,0.931667
2,"INTRUST BANK, NATIONAL ASSOCIATION",557858,0.036783,90,1,6791129,0.107002,False,0.893333,0.956667,0.925000
3,"CHARLES SCHWAB BANK, SSB",3150447,0.043238,87,5,247664000,0.129668,True,0.910000,0.938333,0.924167
4,SOUTHERN BANK AND TRUST COMPANY,717924,0.031172,84,1,4943138,0.104099,False,0.890000,0.918333,0.904167
5,TRUSTCO BANK,677176,0.008521,92,1,6347651,0.104928,False,0.803333,0.978333,0.890833
6,EAST WEST BANK,197478,0.037375,79,1,77620355,0.098608,False,0.896667,0.883333,0.890000
7,WOODFOREST NATIONAL BANK,412751,0.201734,60,1,9113582,0.088460,False,0.976667,0.775000,0.875833
8,UBS BANK USA,3212149,0.375422,58,3,114294933,0.090138,True,0.990000,0.760000,0.875000
9,RAYMOND JAMES BANK,2193616,0.060129,68,1,42320219,0.080971,False,0.926667,0.813333,0.870000


In [111]:
# --------------------------------------------------
# High-distress + high-network candidates
# --------------------------------------------------

score_df["high_distress"] = (
    score_df["distress_percentile"] >= 0.90
)

score_df["high_network"] = (
    score_df["network_percentile"] >= 0.90
)

high_high = (
    score_df[
        score_df["high_distress"] &
        score_df["high_network"]
    ]
    [
        [
            "Financial Institution Name",
            "IDRSSD",
            "failure_probability",
            "network_degree",
            "banks_represented",
            "total_assets",
            "t1crc",
            "is_mds",
            "systemic_risk_score"
        ]
    ]
    .sort_values(
        "systemic_risk_score",
        ascending=False
    )
)

print("High-distress + high-network banks:",
      len(high_high))

high_high

High-distress + high-network banks: 2


,Financial Institution Name,IDRSSD,failure_probability,network_degree,banks_represented,total_assets,t1crc,is_mds,systemic_risk_score
148889,FIRSTBANK,288853,0.170207,95,1,26761023,0.089818,False,0.985833
146775,"CHARLES SCHWAB BANK, SSB",3150447,0.043238,87,5,247664000,0.129668,True,0.924167


In [116]:
from pathlib import Path

FFIEC_ROOT = Path(
    "/Users/irfanhussain/Desktop/systemic-risk-project/data/raw/ffiec"
)

print("All files under FFIEC:\n")

for p in sorted(FFIEC_ROOT.rglob("*")):
    if p.is_file() and not p.name.startswith("."):
        print(p)

All files under FFIEC:

/Users/irfanhussain/Desktop/systemic-risk-project/data/raw/ffiec/2018/part1.txt
/Users/irfanhussain/Desktop/systemic-risk-project/data/raw/ffiec/2018/part2.txt
/Users/irfanhussain/Desktop/systemic-risk-project/data/raw/ffiec/2019/part1.txt
/Users/irfanhussain/Desktop/systemic-risk-project/data/raw/ffiec/2019/part2.txt
/Users/irfanhussain/Desktop/systemic-risk-project/data/raw/ffiec/2020/part1.txt
/Users/irfanhussain/Desktop/systemic-risk-project/data/raw/ffiec/2020/part2.txt
/Users/irfanhussain/Desktop/systemic-risk-project/data/raw/ffiec/2021/part1.txt
/Users/irfanhussain/Desktop/systemic-risk-project/data/raw/ffiec/2021/part2.txt
/Users/irfanhussain/Desktop/systemic-risk-project/data/raw/ffiec/2022/part1.txt
/Users/irfanhussain/Desktop/systemic-risk-project/data/raw/ffiec/2022/part2.txt
/Users/irfanhussain/Desktop/systemic-risk-project/data/raw/ffiec/2023/part1.txt
/Users/irfanhussain/Desktop/systemic-risk-project/data/raw/ffiec/2023/part2.txt
/Users/irfanhuss

In [118]:
# --------------------------------------------------
# Broader search for possible exposure variables
# --------------------------------------------------

patterns = [
    "deposit",
    "depository",
    "interbank",
    "federal",
    "funds",
    "correspondent",
    "due",
    "other banks",
    "other depository",
    "financial institution",
    "bank"
]

for part in ["part1.txt", "part2.txt"]:

    path = FFIEC_2025 / part

    meta = pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        nrows=2,
        header=None
    )

    codes = meta.iloc[0].astype("string").str.strip()
    descriptions = meta.iloc[1].astype("string").str.strip()

    mask = descriptions.str.contains(
        "|".join(patterns),
        case=False,
        na=False,
        regex=True
    )

    result = pd.DataFrame({
        "code": codes[mask].values,
        "description": descriptions[mask].values
    })

    print("\n" + "=" * 100)
    print(part)
    print("=" * 100)

    print(result.to_string(index=False))


part1.txt
    code                          description
RCFD0071   INT-BEARING BALS DUE FRM DEP INSTS
RCFD1406 TOTAL, PAST DUE 30-89 DAYS, ACCRUING
RCFD1407 TOTAL, PAST DUE 90 OR MORE, ACCRUING
RCFD3505 OTH ASSETS-PAST DUE 30-89 DYS, ACCRG
RCFD3506 OTH ASSETS-PAST DUE 90DYS OR MOR,ACG
RCFD6558 COMMCL RE,ETC.-PAST DUE 30-89 DYS,AG
RCFD6559 COMMCL RE,ETC.-PAST DUE 90DYS OR MOR
RCFDB575 PAST DUE(30-89DA): CREDIT CARD LOANS
RCFDB576 PAST DUE(>=90DAY): CREDIT CARD LOANS
RCFDB577 PAST DUE(NONACCR): CREDIT CARD LOANS
RCFDF169   ALL OTHR LEASE PAST DUE 30-89 DAYS
RCFDF170    ALL OTHR LEASE PAST DUE > 90 DAYS
RCFDPV25 Loans to private equity funds -- Dom
RCFN2200                       TOTAL DEPOSITS
RCFN6631        NON-INTEREST BEARING DEPOSITS
RCFNB572 PAST DUE (30-89DA): LOANS FGN OFFICE
RCFNB573 PAST DUE (>=90DAY): LOANS FGN OFFICE
RCFNB574 PAST DUE (NONACCR): LOANS FGN OFFICE
RCON0071   INT-BEARING BALS DUE FRM DEP INSTS
RCON1406 TOTAL, PAST DUE 30-89 DAYS, ACCRUING
RCON1407 TOTAL, PAST DU

In [119]:
# --------------------------------------------------
# Search for likely bilateral/aggregate exposure fields
# --------------------------------------------------

exposure_words = [
    "due from",
    "due to",
    "balances due",
    "balances with",
    "owed to",
    "owed from",
    "claims on",
    "amounts due",
    "federal funds sold",
    "federal funds purchased",
    "depository institutions",
    "other banks"
]

for part in ["part1.txt", "part2.txt"]:

    path = FFIEC_2025 / part

    meta = pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        nrows=2,
        header=None
    )

    codes = meta.iloc[0].astype("string").str.strip()
    descriptions = meta.iloc[1].astype("string").str.strip()

    mask = descriptions.str.contains(
        "|".join(exposure_words),
        case=False,
        na=False,
        regex=True
    )

    result = pd.DataFrame({
        "code": codes[mask].values,
        "description": descriptions[mask].values
    })

    print("\n" + "=" * 100)
    print(part)
    print("=" * 100)
    print(result.to_string(index=False))


part1.txt
Empty DataFrame
Columns: [code, description]
Index: []

part2.txt
Empty DataFrame
Columns: [code, description]
Index: []


In [120]:
# --------------------------------------------------
# Extract aggregate interbank exposure variables
# --------------------------------------------------

RC_2025_PATH = (
    FFIEC_2025 / "part1.txt"
)

exposure_cols = [
    "IDRSSD",
    "Financial Institution Name",
    "RCFD0071",
    "RCON0071",
    "RCONB987",
    "RCONB993"
]

rc_exp = pd.read_csv(
    RC_2025_PATH,
    sep="\t",
    dtype=str,
    usecols=exposure_cols
)

# Remove metadata row if present
if len(rc_exp) > 0 and pd.isna(rc_exp.iloc[0]["IDRSSD"]):
    rc_exp = rc_exp.iloc[1:].copy()

# Clean ID
rc_exp["IDRSSD"] = (
    rc_exp["IDRSSD"]
    .astype("string")
    .str.strip()
)

# Convert numeric fields
for col in exposure_cols[2:]:
    rc_exp[col] = pd.to_numeric(
        rc_exp[col],
        errors="coerce"
    )

# Combine reporting-basis fields
rc_exp["interbank_assets"] = (
    rc_exp["RCFD0071"]
    .combine_first(rc_exp["RCON0071"])
)

# Keep federal funds sold/purchased
rc_exp["fed_funds_sold"] = rc_exp["RCONB987"]
rc_exp["fed_funds_purchased"] = rc_exp["RCONB993"]

print("Rows:", len(rc_exp))

print("\nExposure coverage:")
print(
    rc_exp[
        [
            "interbank_assets",
            "fed_funds_sold",
            "fed_funds_purchased"
        ]
    ].notna().sum()
)

Rows: 17825

Exposure coverage:
interbank_assets       17825
fed_funds_sold         17825
fed_funds_purchased    17825
dtype: int64


In [122]:
# --------------------------------------------------
# Diagnose duplicate IDRSSD in exposure data
# --------------------------------------------------

print(
    "Total rows in rc_exp:",
    len(rc_exp)
)

print(
    "Unique IDRSSD:",
    rc_exp["IDRSSD"].nunique()
)

dup_counts = (
    rc_exp["IDRSSD"]
    .value_counts()
)

print("\nIDRSSDs appearing more than once:")
print(
    dup_counts[
        dup_counts > 1
    ].head(20)
)

print(
    "\nNumber of duplicated IDRSSDs:",
    (dup_counts > 1).sum()
)

Total rows in rc_exp: 17825
Unique IDRSSD: 4522

IDRSSDs appearing more than once:
IDRSSD
37        4
836843    4
837541    4
837448    4
837354    4
837158    4
837149    4
837046    4
836656    4
841753    4
835770    4
835743    4
835659    4
835613    4
835444    4
835378    4
837608    4
837653    4
837840    4
837952    4
Name: count, dtype: Int64

Number of duplicated IDRSSDs: 4477


In [123]:
# --------------------------------------------------
# Inspect duplicated records
# --------------------------------------------------

duplicate_ids = dup_counts[
    dup_counts > 1
].index

rc_exp[
    rc_exp["IDRSSD"].isin(duplicate_ids)
].sort_values("IDRSSD").head(30)

,IDRSSD,Financial Institution Name,RCFD0071,RCON0071,RCONB987,RCONB993,interbank_assets,fed_funds_sold,fed_funds_purchased
7951,1000052,WASHINGTON COUNTY BANK,NaN,1162.0,0,0,1162.0,0,0
12417,1000052,WASHINGTON COUNTY BANK,NaN,2831.0,0,0,2831.0,0,0
3529,1000052,WASHINGTON COUNTY BANK,NaN,1104.0,0,1900,1104.0,0,1900
16926,1000052,WASHINGTON COUNTY BANK,NaN,1526.0,0,0,1526.0,0,0
12418,1000276,PITTSFIELD CO-OPERATIVE BANK,NaN,27590.0,1432,0,27590.0,1432,0
7952,1000276,PITTSFIELD CO-OPERATIVE BANK,NaN,5115.0,2344,0,5115.0,2344,0
16927,1000276,PITTSFIELD CO-OPERATIVE BANK,NaN,6656.0,925,0,6656.0,925,0
3530,1000276,PITTSFIELD CO-OPERATIVE BANK,NaN,11277.0,0,0,11277.0,0,0
12419,1000641,CITIZENS STATE BANK OF NEW CASTLE,NaN,28470.0,0,0,28470.0,0,0
7953,1000641,CITIZENS STATE BANK OF NEW CASTLE,NaN,7295.0,0,0,7295.0,0,0


In [124]:
print(
    "Unique IDs in top_banks:",
    top_banks["IDRSSD"].nunique()
)

print(
    "Rows in top_banks:",
    len(top_banks)
)

Unique IDs in top_banks: 300
Rows in top_banks: 300


In [125]:
# --------------------------------------------------
# Correctly extract 2025-06-30 exposure data
# --------------------------------------------------

RC_2025_PATH = (
    FFIEC_2025 / "part1.txt"
)

exposure_cols = [
    "IDRSSD",
    "Reporting Period End Date",
    "RCFD0071",
    "RCON0071",
    "RCONB987",
    "RCONB993"
]

rc_exp = pd.read_csv(
    RC_2025_PATH,
    sep="\t",
    dtype=str,
    usecols=exposure_cols
)

# Remove metadata row if present
if len(rc_exp) > 0 and pd.isna(rc_exp.iloc[0]["IDRSSD"]):
    rc_exp = rc_exp.iloc[1:].copy()

# Clean ID
rc_exp["IDRSSD"] = (
    rc_exp["IDRSSD"]
    .astype("string")
    .str.strip()
)

# Parse reporting date
rc_exp["Reporting Period End Date"] = pd.to_datetime(
    rc_exp["Reporting Period End Date"],
    errors="coerce"
)

# Keep ONLY 2025-06-30
rc_exp = rc_exp[
    rc_exp["Reporting Period End Date"]
    == pd.Timestamp("2025-06-30")
].copy()

# Convert exposure fields
for col in [
    "RCFD0071",
    "RCON0071",
    "RCONB987",
    "RCONB993"
]:
    rc_exp[col] = pd.to_numeric(
        rc_exp[col],
        errors="coerce"
    )

# Combine reporting-basis fields
rc_exp["interbank_assets"] = (
    rc_exp["RCFD0071"]
    .combine_first(rc_exp["RCON0071"])
)

rc_exp["fed_funds_sold"] = rc_exp["RCONB987"]
rc_exp["fed_funds_purchased"] = rc_exp["RCONB993"]

print("Rows after 2025-06-30 filter:", len(rc_exp))
print("Unique banks:", rc_exp["IDRSSD"].nunique())

print(
    "\nDuplicate bank IDs:",
    rc_exp["IDRSSD"].duplicated().sum()
)

Rows after 2025-06-30 filter: 4477
Unique banks: 4477

Duplicate bank IDs: 0


In [126]:
# --------------------------------------------------
# Restrict to the 300-bank universe
# --------------------------------------------------

exposure_300 = rc_exp[
    rc_exp["IDRSSD"].isin(selected_banks)
].copy()

print(
    "Banks matched:",
    exposure_300["IDRSSD"].nunique()
)

print(
    "Rows:",
    len(exposure_300)
)

Banks matched: 300
Rows: 300


In [127]:
# --------------------------------------------------
# Exposure coverage
# --------------------------------------------------

print(
    exposure_300[
        [
            "interbank_assets",
            "fed_funds_sold",
            "fed_funds_purchased"
        ]
    ].notna().sum()
)

print("\nZero counts:")

for col in [
    "interbank_assets",
    "fed_funds_sold",
    "fed_funds_purchased"
]:
    print(
        col,
        (exposure_300[col] == 0).sum()
    )

interbank_assets       300
fed_funds_sold         300
fed_funds_purchased    300
dtype: int64

Zero counts:
interbank_assets 0
fed_funds_sold 235
fed_funds_purchased 248


In [128]:
# --------------------------------------------------
# Exposure distributions
# --------------------------------------------------

exposure_300[
    [
        "interbank_assets",
        "fed_funds_sold",
        "fed_funds_purchased"
    ]
].describe()

,interbank_assets,fed_funds_sold,fed_funds_purchased
count,3.000000e+02,300.000000,3.000000e+02
mean,7.544457e+06,2593.116667,4.811652e+04
std,3.510140e+07,18261.632801,2.112336e+05
min,9.591000e+03,0.000000,0.000000e+00
25%,1.826712e+05,0.000000,0.000000e+00
50%,4.812585e+05,0.000000,0.000000e+00
75%,1.348672e+06,0.000000,0.000000e+00
max,3.970750e+08,298043.000000,1.978200e+06


In [131]:
# --------------------------------------------------
# Attach total assets to exposure_300
# --------------------------------------------------

asset_map = (
    top_banks
    .set_index("IDRSSD")["total_assets"]
    .astype(float)
    .to_dict()
)

exposure_300["total_assets"] = (
    exposure_300["IDRSSD"].map(asset_map)
)

print(
    "Missing total_assets:",
    exposure_300["total_assets"].isna().sum()
)

print(
    "Rows:",
    len(exposure_300)
)

Missing total_assets: 0
Rows: 300


In [132]:
# --------------------------------------------------
# Interbank exposure intensity
# --------------------------------------------------

exposure_300["interbank_asset_ratio"] = (
    exposure_300["interbank_assets"]
    / exposure_300["total_assets"]
)

exposure_300["fed_funds_sold_ratio"] = (
    exposure_300["fed_funds_sold"]
    / exposure_300["total_assets"]
)

exposure_300["fed_funds_purchased_ratio"] = (
    exposure_300["fed_funds_purchased"]
    / exposure_300["total_assets"]
)

print(
    exposure_300[
        [
            "interbank_asset_ratio",
            "fed_funds_sold_ratio",
            "fed_funds_purchased_ratio"
        ]
    ].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

       interbank_asset_ratio  fed_funds_sold_ratio  fed_funds_purchased_ratio
count             300.000000            300.000000                 300.000000
mean                0.061956              0.000199                   0.002115
std                 0.087696              0.001412                   0.009734
min                 0.001203              0.000000                   0.000000
50%                 0.036054              0.000000                   0.000000
75%                 0.069225              0.000000                   0.000000
90%                 0.134653              0.000120                   0.003836
95%                 0.195776              0.000664                   0.011563
99%                 0.414776              0.003212                   0.039891
max                 0.967413              0.022369                   0.092023


In [133]:
# --------------------------------------------------
# Largest interbank-asset positions
# --------------------------------------------------

exposure_300[
    [
        "IDRSSD",
        "interbank_assets",
        "total_assets",
        "interbank_asset_ratio"
    ]
].sort_values(
    "interbank_assets",
    ascending=False
).head(20)

,IDRSSD,interbank_assets,total_assets,interbank_asset_ratio
11861,852218,397075000.0,3.788551e+09,0.104809
10511,476810,297873000.0,1.833933e+09,0.162423
10527,480228,220983000.0,2.665555e+09,0.082903
10416,451965,159937000.0,1.746394e+09,0.091581
10748,541101,145342000.0,3.982930e+08,0.364912
8948,35301,118289000.0,3.717840e+08,0.318166
12610,2182786,93918000.0,6.254100e+08,0.150170
9548,210434,59601514.0,1.713099e+08,0.347916
9222,112837,52989528.0,6.489094e+08,0.081659
12539,1394676,51815758.0,2.117210e+08,0.244736


In [134]:
# --------------------------------------------------
# Synthetic bilateral interbank exposure matrix
# --------------------------------------------------

banks = (
    top_banks[
        ["IDRSSD", "total_assets"]
    ]
    .copy()
    .sort_values("IDRSSD")
    .reset_index(drop=True)
)

bank_ids = banks["IDRSSD"].tolist()

assets = (
    banks["total_assets"]
    .astype(float)
    .values
)

n = len(bank_ids)

# Observed aggregate interbank assets
ib_map = (
    exposure_300
    .set_index("IDRSSD")["interbank_assets"]
    .astype(float)
    .to_dict()
)

W = np.zeros((n, n), dtype=float)

for i, bank_i in enumerate(bank_ids):

    target_exposure = ib_map[bank_i]

    # Borrower-size weights
    weights = assets.copy()

    # No self-exposure
    weights[i] = 0.0

    # Normalize
    weights = weights / weights.sum()

    # Allocate lender's observed interbank assets
    W[i, :] = target_exposure * weights

    # Explicitly remove self exposure
    W[i, i] = 0.0

print("Exposure matrix shape:", W.shape)
print("Diagonal maximum:", np.diag(W).max())
print("Total reconstructed exposure:", W.sum())

Exposure matrix shape: (300, 300)
Diagonal maximum: 0.0
Total reconstructed exposure: 2263337171.0


In [135]:
# --------------------------------------------------
# Verify reconstructed row sums
# --------------------------------------------------

observed = np.array([
    ib_map[bank_id]
    for bank_id in bank_ids
])

reconstructed = W.sum(axis=1)

verification = pd.DataFrame({
    "IDRSSD": bank_ids,
    "observed_interbank_assets": observed,
    "reconstructed_total": reconstructed
})

verification["absolute_error"] = (
    verification["observed_interbank_assets"]
    - verification["reconstructed_total"]
).abs()

verification["relative_error"] = np.where(
    verification["observed_interbank_assets"] > 0,
    verification["absolute_error"]
    / verification["observed_interbank_assets"],
    0
)

print(
    verification.sort_values(
        "absolute_error",
        ascending=False
    ).head(10)
)

print(
    "\nMaximum absolute error:",
    verification["absolute_error"].max()
)

print(
    "Maximum relative error:",
    verification["relative_error"].max()
)

print(
    "Observed total:",
    observed.sum()
)

print(
    "Reconstructed total:",
    reconstructed.sum()
)

      IDRSSD  observed_interbank_assets  reconstructed_total  absolute_error  \
48   2182786                 93918000.0           93918000.0    1.490116e-08   
43    210434                 59601514.0           59601514.0    7.450581e-09   
22   1394676                 51815758.0           51815758.0    7.450581e-09   
46    214807                 17758000.0           17758000.0    3.725290e-09   
260   817824                 24445285.0           24445285.0    3.725290e-09   
54    233031                  7699000.0            7699000.0    1.862645e-09   
88    280110                 11529432.0           11529432.0    1.862645e-09   
116  3303298                  7504424.0            7504424.0    9.313226e-10   
93   2917317                  1871803.0            1871803.0    2.328306e-10   
35   1893049                  1905000.0            1905000.0    2.328306e-10   

     relative_error  
48     1.586614e-16  
43     1.250066e-16  
22     1.437899e-16  
46     2.097810e-16  
260    1.

In [136]:
# --------------------------------------------------
# Largest reconstructed bilateral exposures
# --------------------------------------------------

rows = []

for i, lender in enumerate(bank_ids):

    for j, borrower in enumerate(bank_ids):

        if i != j and W[i, j] > 0:

            rows.append({
                "lender": lender,
                "borrower": borrower,
                "exposure": W[i, j]
            })

bilateral_df = pd.DataFrame(rows)

# Add names
name_map = (
    top_banks
    .set_index("IDRSSD")[
        "Financial Institution Name"
    ]
    .to_dict()
)

bilateral_df["lender_name"] = (
    bilateral_df["lender"].map(name_map)
)

bilateral_df["borrower_name"] = (
    bilateral_df["borrower"].map(name_map)
)

bilateral_df.sort_values(
    "exposure",
    ascending=False
).head(20)

,lender,borrower,exposure,lender_name,borrower_name
79710,852218,480228,5.684690e+07,"JPMORGAN CHASE BANK, NATIONAL ASSOCIATION","BANK OF AMERICA, NATIONAL ASSOCIATION"
52291,476810,852218,5.485248e+07,"CITIBANK, N.A.","JPMORGAN CHASE BANK, NATIONAL ASSOCIATION"
52889,480228,852218,4.240760e+07,"BANK OF AMERICA, NATIONAL ASSOCIATION","JPMORGAN CHASE BANK, NATIONAL ASSOCIATION"
79708,852218,476810,3.911133e+07,"JPMORGAN CHASE BANK, NATIONAL ASSOCIATION","CITIBANK, N.A."
52201,476810,480228,3.859320e+07,"CITIBANK, N.A.","BANK OF AMERICA, NATIONAL ASSOCIATION"
79699,852218,451965,3.724444e+07,"JPMORGAN CHASE BANK, NATIONAL ASSOCIATION","WELLS FARGO BANK, NATIONAL ASSOCIATION"
49600,451965,852218,2.932717e+07,"WELLS FARGO BANK, NATIONAL ASSOCIATION","JPMORGAN CHASE BANK, NATIONAL ASSOCIATION"
52191,476810,451965,2.528514e+07,"CITIBANK, N.A.","WELLS FARGO BANK, NATIONAL ASSOCIATION"
57075,541101,852218,2.501851e+07,"BANK OF NEW YORK MELLON, THE","JPMORGAN CHASE BANK, NATIONAL ASSOCIATION"
49510,451965,480228,2.063406e+07,"WELLS FARGO BANK, NATIONAL ASSOCIATION","BANK OF AMERICA, NATIONAL ASSOCIATION"


In [137]:
# --------------------------------------------------
# Directed weighted contagion graph
# --------------------------------------------------

G_contagion = nx.DiGraph()

for i, bank_id in enumerate(bank_ids):

    G_contagion.add_node(
        bank_id,
        total_assets=assets[i],
        name=name_map[bank_id]
    )

for i, lender in enumerate(bank_ids):

    for j, borrower in enumerate(bank_ids):

        if i != j and W[i, j] > 0:

            G_contagion.add_edge(
                lender,
                borrower,
                exposure=float(W[i, j])
            )

print("Nodes:", G_contagion.number_of_nodes())
print("Edges:", G_contagion.number_of_edges())

total_graph_exposure = sum(
    d["exposure"]
    for _, _, d in G_contagion.edges(data=True)
)

print(
    "Total graph exposure:",
    total_graph_exposure
)

Nodes: 300
Edges: 89700
Total graph exposure: 2263337171.0


In [138]:
# --------------------------------------------------
# Tier-1 capital for contagion simulation
# --------------------------------------------------

t1_map = (
    top_banks
    .assign(
        tier1_capital=lambda df:
            df["t1crc"] *
            df["total_assets"]
    )
    .set_index("IDRSSD")["tier1_capital"]
    .astype(float)
    .to_dict()
)

for bank_id in G_contagion.nodes():

    G_contagion.nodes[bank_id][
        "tier1_capital"
    ] = t1_map[bank_id]

print(
    "Tier-1 capital attached:",
    len(t1_map)
)

Tier-1 capital attached: 300


In [139]:
# --------------------------------------------------
# First-round contagion simulation
# --------------------------------------------------

def simulate_first_round(
    shocked_bank,
    W,
    bank_ids,
    tier1_map,
    lgd=1.0
):
    """
    Shock one borrower to default.

    lender i loses:
        LGD * W[i, shocked_bank]
    """

    index = {
        bank_id: i
        for i, bank_id in enumerate(bank_ids)
    }

    shocked_idx = index[shocked_bank]

    losses = (
        lgd *
        W[:, shocked_idx]
    )

    result = pd.DataFrame({
        "IDRSSD": bank_ids,
        "loss": losses,
        "tier1_capital": [
            tier1_map[b]
            for b in bank_ids
        ]
    })

    result["loss_to_tier1"] = (
        result["loss"]
        / result["tier1_capital"]
    )

    result["fails"] = (
        result["loss_to_tier1"] >= 1.0
    )

    return result

In [140]:
# --------------------------------------------------
# Shock largest bank
# --------------------------------------------------

largest_bank = (
    top_banks
    .sort_values(
        "total_assets",
        ascending=False
    )
    .iloc[0]["IDRSSD"]
)

shock_result = simulate_first_round(
    largest_bank,
    W,
    bank_ids,
    t1_map,
    lgd=1.0
)

print(
    "Shocked bank:",
    name_map[largest_bank]
)

print(
    "First-round failures:",
    shock_result["fails"].sum()
)

shock_result[
    shock_result["loss"] > 0
].sort_values(
    "loss_to_tier1",
    ascending=False
).head(20)

Shocked bank: JPMORGAN CHASE BANK, NATIONAL ASSOCIATION
First-round failures: 2


,IDRSSD,loss,tier1_capital,loss_to_tier1,fails
187,52719,9.253450e+05,863447.0,1.071687,True
190,541101,2.501851e+07,23901000.0,1.046756,True
132,35301,2.033723e+07,20572000.0,0.988588,False
43,210434,1.015480e+07,10278574.0,0.987958,False
59,243375,3.909555e+05,406732.0,0.961212,False
286,934329,9.986959e+05,1993000.0,0.501102,False
22,1394676,8.844354e+06,18181353.0,0.486452,False
20,1386251,5.592805e+05,1225694.0,0.456297,False
278,918918,4.771286e+05,1059615.0,0.450285,False
53,229913,5.377641e+05,1294284.0,0.415492,False


In [141]:
# --------------------------------------------------
# Multi-round contagion simulator
# --------------------------------------------------

def simulate_cascade(
    shocked_bank,
    W,
    bank_ids,
    tier1_map,
    lgd=1.0,
    max_rounds=20
):
    """
    Simulate sequential contagion after one bank defaults.

    W[i, j] = exposure of lender i to borrower j.

    A bank fails when cumulative losses >= Tier-1 capital.
    """

    index = {
        bank_id: i
        for i, bank_id in enumerate(bank_ids)
    }

    shock_idx = index[shocked_bank]

    n = len(bank_ids)

    # Cumulative losses suffered by each bank
    cumulative_losses = np.zeros(n)

    # Initially only the shocked bank has failed
    failed = np.zeros(n, dtype=bool)
    failed[shock_idx] = True

    # Banks that failed in the latest round
    newly_failed = np.array(
        [shock_idx],
        dtype=int
    )

    rounds = 0

    while (
        len(newly_failed) > 0
        and rounds < max_rounds
    ):

        rounds += 1

        # Losses caused by banks that failed
        # in the previous round
        round_losses = (
            lgd *
            W[:, newly_failed].sum(axis=1)
        )

        cumulative_losses += round_losses

        # Only currently surviving banks can newly fail
        candidates = ~failed

        tier1 = np.array([
            tier1_map[b]
            for b in bank_ids
        ])

        new_failures = (
            candidates &
            (cumulative_losses >= tier1)
        )

        newly_failed = np.where(
            new_failures
        )[0]

        failed[newly_failed] = True

    failed_ids = [
        bank_ids[i]
        for i in np.where(failed)[0]
    ]

    return {
        "shocked_bank": shocked_bank,
        "failed_banks": failed_ids,
        "total_failed": int(failed.sum()),
        "secondary_failures": int(failed.sum() - 1),
        "rounds": rounds,
        "total_cumulative_loss": float(
            cumulative_losses.sum()
        ),
        "max_loss_to_tier1": float(
            np.max(
                cumulative_losses / tier1
            )
        )
    }

In [142]:
# --------------------------------------------------
# Multi-round JPMorgan stress
# --------------------------------------------------

jpm_result = simulate_cascade(
    shocked_bank=largest_bank,
    W=W,
    bank_ids=bank_ids,
    tier1_map=t1_map,
    lgd=1.0
)

print("Initial shock:",
      name_map[jpm_result["shocked_bank"]])

print("Total failed:",
      jpm_result["total_failed"])

print("Secondary failures:",
      jpm_result["secondary_failures"])

print("Contagion rounds:",
      jpm_result["rounds"])

print(
    "Total cumulative loss:",
    jpm_result["total_cumulative_loss"]
)

Initial shock: JPMORGAN CHASE BANK, NATIONAL ASSOCIATION
Total failed: 6
Secondary failures: 5
Contagion rounds: 3
Total cumulative loss: 428238398.42279005


In [143]:
# --------------------------------------------------
# Systemic impact of every possible initial shock
# --------------------------------------------------

cascade_results = []

for bank in bank_ids:

    result = simulate_cascade(
        shocked_bank=bank,
        W=W,
        bank_ids=bank_ids,
        tier1_map=t1_map,
        lgd=1.0
    )

    cascade_results.append(result)

cascade_df = pd.DataFrame(cascade_results)

# Add readable bank names
cascade_df["Financial Institution Name"] = (
    cascade_df["shocked_bank"].map(name_map)
)

# Put useful columns first
cascade_df = cascade_df[
    [
        "shocked_bank",
        "Financial Institution Name",
        "total_failed",
        "secondary_failures",
        "rounds",
        "total_cumulative_loss",
        "max_loss_to_tier1"
    ]
].sort_values(
    [
        "total_failed",
        "total_cumulative_loss"
    ],
    ascending=False
).reset_index(drop=True)

cascade_df.head(25)

,shocked_bank,Financial Institution Name,total_failed,secondary_failures,rounds,total_cumulative_loss,max_loss_to_tier1
0,852218,"JPMORGAN CHASE BANK, NATIONAL ASSOCIATION",6,5,3,4.282384e+08,1.339349
1,480228,"BANK OF AMERICA, NATIONAL ASSOCIATION",1,0,1,2.592550e+08,0.754019
2,451965,"WELLS FARGO BANK, NATIONAL ASSOCIATION",1,0,1,1.758860e+08,0.494012
3,476810,"CITIBANK, N.A.",1,0,1,1.723463e+08,0.518774
4,504713,U.S. BANK NATIONAL ASSOCIATION,1,0,1,7.123317e+07,0.189933
5,112837,"CAPITAL ONE, NATIONAL ASSOCIATION",1,0,1,6.879703e+07,0.183560
6,2182786,GOLDMAN SACHS BANK USA,1,0,1,6.513213e+07,0.176913
7,817824,"PNC BANK, NATIONAL ASSOCIATION",1,0,1,5.952576e+07,0.156875
8,852320,TRUIST BANK,1,0,1,5.719969e+07,0.151504
9,541101,"BANK OF NEW YORK MELLON, THE",1,0,1,4.056658e+07,0.112667


In [144]:
# --------------------------------------------------
# Add network information
# --------------------------------------------------

cascade_df["is_mds"] = (
    cascade_df["shocked_bank"].isin(mds_set)
)

cascade_df["total_assets"] = (
    cascade_df["shocked_bank"].map(asset_map)
)

cascade_df["network_degree"] = (
    cascade_df["shocked_bank"].map(degree_map)
)

cascade_df["failure_probability"] = (
    cascade_df["shocked_bank"].map(
        risk_snapshot
        .set_index("IDRSSD")["failure_probability"]
    )
)

cascade_df[
    [
        "Financial Institution Name",
        "shocked_bank",
        "total_assets",
        "network_degree",
        "is_mds",
        "failure_probability",
        "secondary_failures",
        "total_cumulative_loss",
        "rounds"
    ]
].head(25)

,Financial Institution Name,shocked_bank,total_assets,network_degree,is_mds,failure_probability,secondary_failures,total_cumulative_loss,rounds
0,"JPMORGAN CHASE BANK, NATIONAL ASSOCIATION",852218,3.788551e+09,0,True,0.000971,5,4.282384e+08,3
1,"BANK OF AMERICA, NATIONAL ASSOCIATION",480228,2.665555e+09,2,False,0.006026,0,2.592550e+08,1
2,"WELLS FARGO BANK, NATIONAL ASSOCIATION",451965,1.746394e+09,0,True,0.000148,0,1.758860e+08,1
3,"CITIBANK, N.A.",476810,1.833933e+09,0,True,0.000933,0,1.723463e+08,1
4,U.S. BANK NATIONAL ASSOCIATION,504713,6.714381e+08,32,True,0.005403,0,7.123317e+07,1
5,"CAPITAL ONE, NATIONAL ASSOCIATION",112837,6.489094e+08,74,False,0.000653,0,6.879703e+07,1
6,GOLDMAN SACHS BANK USA,2182786,6.254100e+08,0,True,0.000553,0,6.513213e+07,1
7,"PNC BANK, NATIONAL ASSOCIATION",817824,5.545731e+08,67,False,0.000291,0,5.952576e+07,1
8,TRUIST BANK,852320,5.355850e+08,59,False,0.000159,0,5.719969e+07,1
9,"BANK OF NEW YORK MELLON, THE",541101,3.982930e+08,0,True,0.139585,0,4.056658e+07,1


In [145]:
# --------------------------------------------------
# Contagion impact: MDS vs non-MDS
# --------------------------------------------------

cascade_comparison = (
    cascade_df
    .groupby("is_mds")
    .agg(
        banks=("shocked_bank", "size"),
        mean_secondary_failures=(
            "secondary_failures", "mean"
        ),
        median_secondary_failures=(
            "secondary_failures", "median"
        ),
        max_secondary_failures=(
            "secondary_failures", "max"
        ),
        mean_cumulative_loss=(
            "total_cumulative_loss", "mean"
        ),
        max_cumulative_loss=(
            "total_cumulative_loss", "max"
        )
    )
)

cascade_comparison

,banks,mean_secondary_failures,median_secondary_failures,max_secondary_failures,mean_cumulative_loss,max_cumulative_loss
is_mds,,,,,,
False,183,0.000000,0.0,0,5.378393e+06,2.592550e+08
True,117,0.042735,0.0,5,1.177128e+07,4.282384e+08


In [146]:
# --------------------------------------------------
# LGD sensitivity analysis
# --------------------------------------------------

lgd_values = [1.00, 0.75, 0.50, 0.25]

lgd_results = []

for lgd in lgd_values:

    for bank in bank_ids:

        result = simulate_cascade(
            shocked_bank=bank,
            W=W,
            bank_ids=bank_ids,
            tier1_map=t1_map,
            lgd=lgd
        )

        lgd_results.append({
            "lgd": lgd,
            "shocked_bank": bank,
            "total_failed": result["total_failed"],
            "secondary_failures": result["secondary_failures"],
            "rounds": result["rounds"],
            "total_cumulative_loss": result[
                "total_cumulative_loss"
            ]
        })

lgd_df = pd.DataFrame(lgd_results)

# Summarize the most damaging shock at each LGD
lgd_summary = (
    lgd_df
    .groupby("lgd")
    .agg(
        max_total_failed=("total_failed", "max"),
        max_secondary_failures=(
            "secondary_failures", "max"
        ),
        max_rounds=("rounds", "max"),
        max_cumulative_loss=(
            "total_cumulative_loss", "max"
        ),
        shocks_causing_cascade=(
            "secondary_failures",
            lambda x: (x > 0).sum()
        )
    )
    .reset_index()
)

lgd_summary

,lgd,max_total_failed,max_secondary_failures,max_rounds,max_cumulative_loss,shocks_causing_cascade
0,0.25,1,0,1,8.252253e+07,0
1,0.50,1,0,1,1.650451e+08,0
2,0.75,1,0,1,2.475676e+08,0
3,1.00,6,5,3,4.282384e+08,1


In [147]:
# --------------------------------------------------
# Worst initial shock at each LGD
# --------------------------------------------------

worst_by_lgd = (
    lgd_df
    .sort_values(
        [
            "lgd",
            "total_failed",
            "total_cumulative_loss"
        ],
        ascending=[True, False, False]
    )
    .groupby("lgd")
    .head(1)
    .copy()
)

worst_by_lgd["Financial Institution Name"] = (
    worst_by_lgd["shocked_bank"].map(name_map)
)

worst_by_lgd[
    [
        "lgd",
        "Financial Institution Name",
        "total_failed",
        "secondary_failures",
        "rounds",
        "total_cumulative_loss"
    ]
]

,lgd,Financial Institution Name,total_failed,secondary_failures,rounds,total_cumulative_loss
1166,0.25,"JPMORGAN CHASE BANK, NATIONAL ASSOCIATION",1,0,1,8.252253e+07
866,0.50,"JPMORGAN CHASE BANK, NATIONAL ASSOCIATION",1,0,1,1.650451e+08
566,0.75,"JPMORGAN CHASE BANK, NATIONAL ASSOCIATION",1,0,1,2.475676e+08
266,1.00,"JPMORGAN CHASE BANK, NATIONAL ASSOCIATION",6,5,3,4.282384e+08


In [148]:
# --------------------------------------------------
# Normalize contagion impact
# --------------------------------------------------

total_system_assets = assets.sum()

contagion_rank = cascade_df.copy()

contagion_rank["cascade_fraction"] = (
    contagion_rank["secondary_failures"]
    / (len(bank_ids) - 1)
)

contagion_rank["system_loss_fraction"] = (
    contagion_rank["total_cumulative_loss"]
    / total_system_assets
)

contagion_rank = contagion_rank.sort_values(
    "system_loss_fraction",
    ascending=False
).reset_index(drop=True)

contagion_rank[
    [
        "Financial Institution Name",
        "shocked_bank",
        "secondary_failures",
        "cascade_fraction",
        "total_cumulative_loss",
        "system_loss_fraction",
        "max_loss_to_tier1",
        "is_mds"
    ]
].head(20)

,Financial Institution Name,shocked_bank,secondary_failures,cascade_fraction,total_cumulative_loss,system_loss_fraction,max_loss_to_tier1,is_mds
0,"JPMORGAN CHASE BANK, NATIONAL ASSOCIATION",852218,5,0.016722,4.282384e+08,0.019111,1.339349,True
1,"BANK OF AMERICA, NATIONAL ASSOCIATION",480228,0,0.000000,2.592550e+08,0.011570,0.754019,False
2,"WELLS FARGO BANK, NATIONAL ASSOCIATION",451965,0,0.000000,1.758860e+08,0.007849,0.494012,True
3,"CITIBANK, N.A.",476810,0,0.000000,1.723463e+08,0.007691,0.518774,True
4,U.S. BANK NATIONAL ASSOCIATION,504713,0,0.000000,7.123317e+07,0.003179,0.189933,True
5,"CAPITAL ONE, NATIONAL ASSOCIATION",112837,0,0.000000,6.879703e+07,0.003070,0.183560,False
6,GOLDMAN SACHS BANK USA,2182786,0,0.000000,6.513213e+07,0.002907,0.176913,True
7,"PNC BANK, NATIONAL ASSOCIATION",817824,0,0.000000,5.952576e+07,0.002657,0.156875,False
8,TRUIST BANK,852320,0,0.000000,5.719969e+07,0.002553,0.151504,False
9,"BANK OF NEW YORK MELLON, THE",541101,0,0.000000,4.056658e+07,0.001810,0.112667,True


In [149]:
# --------------------------------------------------
# Save dense gravity contagion baseline
# --------------------------------------------------

baseline_path = PROCESSED_DIR / "contagion_gravity_baseline.csv"

contagion_rank.to_csv(
    baseline_path,
    index=False
)

lgd_df.to_csv(
    PROCESSED_DIR / "contagion_lgd_sensitivity.csv",
    index=False
)

print("Saved:")
print(baseline_path)
print(PROCESSED_DIR / "contagion_lgd_sensitivity.csv")

Saved:
/Users/irfanhussain/Desktop/systemic-risk-project/data/processed/contagion_gravity_baseline.csv
/Users/irfanhussain/Desktop/systemic-risk-project/data/processed/contagion_lgd_sensitivity.csv


In [150]:
# --------------------------------------------------
# Sparse reconstructed contagion network
# --------------------------------------------------

import networkx as nx
import numpy as np

np.random.seed(42)

N = len(bank_ids)

# Number of undirected backbone edges
# k controls sparsity
K = 8

# Barabasi-Albert requires:
# 1 <= K < N
G_sparse_base = nx.barabasi_albert_graph(
    n=N,
    m=K,
    seed=42
)

print("Sparse backbone nodes:",
      G_sparse_base.number_of_nodes())

print("Sparse backbone edges:",
      G_sparse_base.number_of_edges())

Sparse backbone nodes: 300
Sparse backbone edges: 2336


In [151]:
# --------------------------------------------------
# Build directed sparse exposure matrix
# --------------------------------------------------

W_sparse = np.zeros(
    (N, N),
    dtype=float
)

# Asset-based borrower weights
asset_weights = assets / assets.sum()

for i, lender in enumerate(bank_ids):

    target_exposure = ib_map[lender]

    # Candidate counterparties are the BA neighbors
    neighbors = list(
        G_sparse_base.neighbors(i)
    )

    if len(neighbors) == 0:
        continue

    # Use borrower asset sizes as weights
    weights = assets[neighbors].astype(float)

    # Normalize
    weight_sum = weights.sum()

    if weight_sum <= 0:
        weights = np.ones(len(neighbors))
    else:
        weights = weights / weight_sum

    # Allocate lender's observed interbank assets
    for neighbor, weight in zip(
        neighbors,
        weights
    ):

        # i -> neighbor
        W_sparse[i, neighbor] = (
            target_exposure * weight
        )

print("Sparse matrix shape:",
      W_sparse.shape)

print(
    "Non-zero exposures:",
    np.count_nonzero(W_sparse)
)

Sparse matrix shape: (300, 300)
Non-zero exposures: 4672


In [152]:
# --------------------------------------------------
# Verify sparse row sums
# --------------------------------------------------

sparse_observed = np.array([
    ib_map[b]
    for b in bank_ids
])

sparse_reconstructed = W_sparse.sum(axis=1)

sparse_error = np.abs(
    sparse_observed -
    sparse_reconstructed
)

print(
    "Maximum absolute error:",
    sparse_error.max()
)

print(
    "Maximum relative error:",
    (
        sparse_error /
        sparse_observed
    ).max()
)

print(
    "Observed total exposure:",
    sparse_observed.sum()
)

print(
    "Sparse reconstructed total:",
    sparse_reconstructed.sum()
)

Maximum absolute error: 1.4901161193847656e-08
Maximum relative error: 3.121238298959585e-16
Observed total exposure: 2263337171.0
Sparse reconstructed total: 2263337171.0


In [153]:
# --------------------------------------------------
# Sparse network statistics
# --------------------------------------------------

G_sparse = nx.from_numpy_array(
    W_sparse,
    create_using=nx.DiGraph
)

G_sparse = nx.relabel_nodes(
    G_sparse,
    {
        i: bank_id
        for i, bank_id in enumerate(bank_ids)
    }
)

print("Nodes:", G_sparse.number_of_nodes())
print("Edges:", G_sparse.number_of_edges())

in_degrees = np.array([
    G_sparse.in_degree(n)
    for n in G_sparse.nodes()
])

out_degrees = np.array([
    G_sparse.out_degree(n)
    for n in G_sparse.nodes()
])

print("\nAverage in-degree:",
      in_degrees.mean())

print("Average out-degree:",
      out_degrees.mean())

print("Maximum in-degree:",
      in_degrees.max())

print("Maximum out-degree:",
      out_degrees.max())

Nodes: 300
Edges: 4672

Average in-degree: 15.573333333333334
Average out-degree: 15.573333333333334
Maximum in-degree: 78
Maximum out-degree: 78


In [154]:
# --------------------------------------------------
# JPMorgan shock on sparse network
# --------------------------------------------------

jpm_sparse = simulate_cascade(
    shocked_bank=largest_bank,
    W=W_sparse,
    bank_ids=bank_ids,
    tier1_map=t1_map,
    lgd=1.0
)

print(
    "Initial shock:",
    name_map[largest_bank]
)

print(
    "Total failed:",
    jpm_sparse["total_failed"]
)

print(
    "Secondary failures:",
    jpm_sparse["secondary_failures"]
)

print(
    "Rounds:",
    jpm_sparse["rounds"]
)

print(
    "Total cumulative loss:",
    jpm_sparse["total_cumulative_loss"]
)

Initial shock: JPMORGAN CHASE BANK, NATIONAL ASSOCIATION
Total failed: 3
Secondary failures: 2
Rounds: 3
Total cumulative loss: 336229502.4954526


In [155]:
# --------------------------------------------------
# Sparse-network systemic impact
# --------------------------------------------------

sparse_results = []

for bank in bank_ids:

    result = simulate_cascade(
        shocked_bank=bank,
        W=W_sparse,
        bank_ids=bank_ids,
        tier1_map=t1_map,
        lgd=1.0
    )

    sparse_results.append({
        "shocked_bank": bank,
        "total_failed": result["total_failed"],
        "secondary_failures": result["secondary_failures"],
        "rounds": result["rounds"],
        "total_cumulative_loss": (
            result["total_cumulative_loss"]
        ),
        "max_loss_to_tier1": (
            result["max_loss_to_tier1"]
        )
    })

sparse_cascade_df = pd.DataFrame(
    sparse_results
)

sparse_cascade_df["Financial Institution Name"] = (
    sparse_cascade_df["shocked_bank"]
    .map(name_map)
)

sparse_cascade_df["is_mds"] = (
    sparse_cascade_df["shocked_bank"]
    .isin(mds_set)
)

sparse_cascade_df["total_assets"] = (
    sparse_cascade_df["shocked_bank"]
    .map(asset_map)
)

sparse_cascade_df["network_degree"] = (
    sparse_cascade_df["shocked_bank"]
    .map(degree_map)
)

sparse_cascade_df["failure_probability"] = (
    sparse_cascade_df["shocked_bank"].map(
        risk_snapshot
        .set_index("IDRSSD")[
            "failure_probability"
        ]
    )
)

sparse_cascade_df = (
    sparse_cascade_df
    .sort_values(
        [
            "total_failed",
            "total_cumulative_loss"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

sparse_cascade_df.head(20)

,shocked_bank,total_failed,secondary_failures,rounds,total_cumulative_loss,max_loss_to_tier1,Financial Institution Name,is_mds,total_assets,network_degree,failure_probability
0,852218,3,2,3,3.362295e+08,2.202931,"JPMORGAN CHASE BANK, NATIONAL ASSOCIATION",True,3.788551e+09,0,0.000971
1,1216022,3,2,2,1.475150e+08,1.941349,SYNCHRONY BANK,False,1.138970e+08,15,0.000773
2,476810,3,2,2,8.815672e+07,1.337462,"CITIBANK, N.A.",True,1.833933e+09,0,0.000933
3,1394676,2,1,2,2.965335e+08,2.202931,AMERICAN EXPRESS NATIONAL BANK,False,2.117210e+08,6,0.004436
4,12311,2,1,2,2.155821e+08,1.406030,"HUNTINGTON NATIONAL BANK, THE",True,2.066454e+08,0,0.000820
5,491224,2,1,2,7.449596e+07,2.387005,FIRST-CITIZENS BANK & TRUST COMPANY,False,2.294120e+08,9,0.000190
6,463735,2,1,2,3.902404e+07,1.487529,HANCOCK WHITNEY BANK,False,3.520509e+07,92,0.000270
7,504713,2,1,2,3.678317e+07,1.344991,U.S. BANK NATIONAL ASSOCIATION,True,6.714381e+08,32,0.005403
8,501105,2,1,2,3.477869e+07,2.190293,MANUFACTURERS AND TRADERS TRUST COMPANY,False,2.110420e+08,67,0.000052
9,128904,2,1,2,3.007282e+07,1.075470,EASTERN BANK,True,2.545295e+07,0,0.002284


In [156]:
# --------------------------------------------------
# Dense vs sparse contagion comparison
# --------------------------------------------------

dense_top = cascade_df.iloc[0]

sparse_top = sparse_cascade_df.iloc[0]

comparison = pd.DataFrame({
    "metric": [
        "Edges",
        "Worst initial shock",
        "Maximum total failures",
        "Maximum secondary failures",
        "Maximum rounds",
        "Maximum cumulative loss"
    ],

    "dense_gravity": [
        G_contagion.number_of_edges(),
        dense_top["Financial Institution Name"],
        cascade_df["total_failed"].max(),
        cascade_df["secondary_failures"].max(),
        cascade_df["rounds"].max(),
        cascade_df["total_cumulative_loss"].max()
    ],

    "sparse_reconstructed": [
        G_sparse.number_of_edges(),
        sparse_top["Financial Institution Name"],
        sparse_cascade_df["total_failed"].max(),
        sparse_cascade_df["secondary_failures"].max(),
        sparse_cascade_df["rounds"].max(),
        sparse_cascade_df["total_cumulative_loss"].max()
    ]
})

comparison

,metric,dense_gravity,sparse_reconstructed
0,Edges,89700,4672
1,Worst initial shock,"JPMORGAN CHASE BANK, NATIONAL ASSOCIATION","JPMORGAN CHASE BANK, NATIONAL ASSOCIATION"
2,Maximum total failures,6,3
3,Maximum secondary failures,5,2
4,Maximum rounds,3,3
5,Maximum cumulative loss,428238398.42279,336229502.495453


In [157]:
# --------------------------------------------------
# Correctly label similarity vs contagion network degree
# --------------------------------------------------

sparse_cascade_df["mtds_degree"] = (
    sparse_cascade_df["shocked_bank"].map(degree_map)
)

cont_in_degree_map = dict(G_sparse.in_degree())
cont_out_degree_map = dict(G_sparse.out_degree())

sparse_cascade_df["contagion_in_degree"] = (
    sparse_cascade_df["shocked_bank"]
    .map(cont_in_degree_map)
)

sparse_cascade_df["contagion_out_degree"] = (
    sparse_cascade_df["shocked_bank"]
    .map(cont_out_degree_map)
)

sparse_cascade_df = sparse_cascade_df.drop(
    columns=["network_degree"],
    errors="ignore"
)

sparse_cascade_df[
    [
        "Financial Institution Name",
        "shocked_bank",
        "mtds_degree",
        "contagion_in_degree",
        "contagion_out_degree",
        "total_failed",
        "total_cumulative_loss"
    ]
].head(20)

,Financial Institution Name,shocked_bank,mtds_degree,contagion_in_degree,contagion_out_degree,total_failed,total_cumulative_loss
0,"JPMORGAN CHASE BANK, NATIONAL ASSOCIATION",852218,0,9,9,3,3.362295e+08
1,SYNCHRONY BANK,1216022,15,59,59,3,1.475150e+08
2,"CITIBANK, N.A.",476810,0,9,9,3,8.815672e+07
3,AMERICAN EXPRESS NATIONAL BANK,1394676,6,40,40,2,2.965335e+08
4,"HUNTINGTON NATIONAL BANK, THE",12311,0,74,74,2,2.155821e+08
5,FIRST-CITIZENS BANK & TRUST COMPANY,491224,9,13,13,2,7.449596e+07
6,HANCOCK WHITNEY BANK,463735,92,14,14,2,3.902404e+07
7,U.S. BANK NATIONAL ASSOCIATION,504713,32,8,8,2,3.678317e+07
8,MANUFACTURERS AND TRADERS TRUST COMPANY,501105,67,9,9,2,3.477869e+07
9,EASTERN BANK,128904,0,60,60,2,3.007282e+07


In [158]:
# --------------------------------------------------
# Sparse-network systemic-risk screening score
# --------------------------------------------------

final_score = sparse_cascade_df.copy()

# Distress percentile
final_score["distress_percentile"] = (
    final_score["failure_probability"]
    .rank(pct=True)
)

# Contagion-impact percentile
final_score["contagion_percentile"] = (
    final_score["total_cumulative_loss"]
    .rank(pct=True)
)

# Equal-weight screening score
final_score["systemic_risk_score"] = (
    0.5 * final_score["distress_percentile"]
    + 0.5 * final_score["contagion_percentile"]
)

# Structural information
final_score["mtds_degree"] = (
    final_score["shocked_bank"].map(degree_map)
)

final_score["is_mds"] = (
    final_score["shocked_bank"].isin(mds_set)
)

final_score = final_score.sort_values(
    "systemic_risk_score",
    ascending=False
).reset_index(drop=True)

In [159]:
# --------------------------------------------------
# Top systemic-risk candidates
# --------------------------------------------------

final_score[
    [
        "Financial Institution Name",
        "shocked_bank",
        "failure_probability",
        "total_cumulative_loss",
        "secondary_failures",
        "mtds_degree",
        "is_mds",
        "systemic_risk_score"
    ]
].head(25)

,Financial Institution Name,shocked_bank,failure_probability,total_cumulative_loss,secondary_failures,mtds_degree,is_mds,systemic_risk_score
0,UBS BANK USA,3212149,0.375422,1.735541e+07,0,58,True,0.945000
1,"NORTHERN TRUST COMPANY, THE",210434,0.487829,1.123870e+07,0,1,False,0.930000
2,RAYMOND JAMES BANK,2193616,0.060129,2.378380e+07,0,68,False,0.925000
3,STATE STREET BANK AND TRUST COMPANY,35301,0.123504,1.506021e+07,0,0,True,0.920000
4,"FIRST AMERICAN TRUST, FSB",1883321,0.027101,2.622908e+07,0,0,True,0.911667
5,"NORTHBROOK BANK & TRUST COMPANY, NATIONAL ASSO...",2938198,0.107070,4.591106e+06,0,18,False,0.856667
6,"MORGAN STANLEY PRIVATE BANK, NATIONAL ASSOCIATION",2489805,0.772845,2.912114e+06,0,0,True,0.855000
7,"CHARLES SCHWAB BANK, SSB",3150447,0.043238,6.435417e+06,0,87,True,0.853333
8,"BANK OF AMERICA, NATIONAL ASSOCIATION",480228,0.006026,2.594902e+07,1,2,False,0.850000
9,U.S. BANK NATIONAL ASSOCIATION,504713,0.005403,3.678317e+07,1,32,True,0.848333


In [160]:
# --------------------------------------------------
# High-distress + high-contagion candidates
# --------------------------------------------------

final_score["high_distress"] = (
    final_score["distress_percentile"] >= 0.90
)

final_score["high_contagion"] = (
    final_score["contagion_percentile"] >= 0.90
)

priority_candidates = final_score[
    final_score["high_distress"] &
    final_score["high_contagion"]
].copy()

priority_candidates[
    [
        "Financial Institution Name",
        "shocked_bank",
        "failure_probability",
        "total_cumulative_loss",
        "secondary_failures",
        "mtds_degree",
        "is_mds",
        "systemic_risk_score"
    ]
].sort_values(
    "systemic_risk_score",
    ascending=False
)

,Financial Institution Name,shocked_bank,failure_probability,total_cumulative_loss,secondary_failures,mtds_degree,is_mds,systemic_risk_score
0,UBS BANK USA,3212149,0.375422,1.735541e+07,0,58,True,0.945
2,RAYMOND JAMES BANK,2193616,0.060129,2.378380e+07,0,68,False,0.925


In [161]:
# --------------------------------------------------
# Dense vs sparse top-10 overlap
# --------------------------------------------------

dense_top10 = set(
    cascade_df
    .nlargest(10, "total_cumulative_loss")
    ["shocked_bank"]
)

sparse_top10 = set(
    sparse_cascade_df
    .nlargest(10, "total_cumulative_loss")
    ["shocked_bank"]
)

overlap = dense_top10.intersection(
    sparse_top10
)

print("Dense top-10:", len(dense_top10))
print("Sparse top-10:", len(sparse_top10))
print("Overlap:", len(overlap))

print("\nCommon banks:")

for bank in overlap:
    print(
        bank,
        name_map.get(bank)
    )

Dense top-10: 10
Sparse top-10: 10
Overlap: 3

Common banks:
852218 JPMORGAN CHASE BANK, NATIONAL ASSOCIATION
112837 CAPITAL ONE, NATIONAL ASSOCIATION
476810 CITIBANK, N.A.


In [162]:
# --------------------------------------------------
# Directed sparse exposure reconstruction
# --------------------------------------------------

np.random.seed(42)

N = len(bank_ids)

# Target number of counterparties per lender
MIN_COUNTERPARTIES = 5
MAX_COUNTERPARTIES = 20

W_directed = np.zeros(
    (N, N),
    dtype=float
)

# Borrower attractiveness:
# larger banks are more likely to receive exposures
borrower_weight = assets / assets.sum()

for i, lender in enumerate(bank_ids):

    target_exposure = ib_map[lender]

    # Number of counterparties depends mildly on lender size
    lender_ratio = (
        assets[i] / assets.max()
    )

    k = int(
        MIN_COUNTERPARTIES
        + lender_ratio *
        (MAX_COUNTERPARTIES - MIN_COUNTERPARTIES)
    )

    k = max(
        MIN_COUNTERPARTIES,
        min(MAX_COUNTERPARTIES, k)
    )

    # Candidate borrowers excluding self
    candidates = np.delete(
        np.arange(N),
        i
    )

    candidate_weights = borrower_weight[
        candidates
    ].copy()

    candidate_weights = (
        candidate_weights /
        candidate_weights.sum()
    )

    # Randomly select counterparties
    chosen = np.random.choice(
        candidates,
        size=k,
        replace=False,
        p=candidate_weights
    )

    chosen_weights = borrower_weight[
        chosen
    ].copy()

    chosen_weights = (
        chosen_weights /
        chosen_weights.sum()
    )

    # Allocate all observed interbank assets
    W_directed[
        i,
        chosen
    ] = (
        target_exposure *
        chosen_weights
    )

print(
    "Matrix shape:",
    W_directed.shape
)

print(
    "Non-zero exposures:",
    np.count_nonzero(W_directed)
)

Matrix shape: (300, 300)
Non-zero exposures: 1552


In [163]:
# --------------------------------------------------
# Verify directed reconstruction
# --------------------------------------------------

directed_observed = np.array([
    ib_map[b]
    for b in bank_ids
])

directed_reconstructed = (
    W_directed.sum(axis=1)
)

directed_error = np.abs(
    directed_observed -
    directed_reconstructed
)

print(
    "Maximum absolute error:",
    directed_error.max()
)

print(
    "Maximum relative error:",
    (
        directed_error /
        directed_observed
    ).max()
)

print(
    "Observed total:",
    directed_observed.sum()
)

print(
    "Reconstructed total:",
    directed_reconstructed.sum()
)

Maximum absolute error: 5.960464477539063e-08
Maximum relative error: 3.9008275376564545e-16
Observed total: 2263337171.0
Reconstructed total: 2263337171.0


In [164]:
# --------------------------------------------------
# Directed-network statistics
# --------------------------------------------------

G_directed = nx.from_numpy_array(
    W_directed,
    create_using=nx.DiGraph
)

G_directed = nx.relabel_nodes(
    G_directed,
    {
        i: bank_id
        for i, bank_id in enumerate(bank_ids)
    }
)

in_degrees = np.array([
    G_directed.in_degree(n)
    for n in G_directed.nodes()
])

out_degrees = np.array([
    G_directed.out_degree(n)
    for n in G_directed.nodes()
])

print("Nodes:", G_directed.number_of_nodes())
print("Edges:", G_directed.number_of_edges())

print("\nAverage in-degree:", in_degrees.mean())
print("Average out-degree:", out_degrees.mean())

print("Maximum in-degree:", in_degrees.max())
print("Maximum out-degree:", out_degrees.max())

print(
    "\nBanks where in-degree != out-degree:",
    np.sum(in_degrees != out_degrees)
)

Nodes: 300
Edges: 1552

Average in-degree: 5.173333333333333
Average out-degree: 5.173333333333333
Maximum in-degree: 186
Maximum out-degree: 20

Banks where in-degree != out-degree: 295


In [165]:
# --------------------------------------------------
# JPMorgan shock: genuinely directed network
# --------------------------------------------------

jpm_directed = simulate_cascade(
    shocked_bank=largest_bank,
    W=W_directed,
    bank_ids=bank_ids,
    tier1_map=t1_map,
    lgd=1.0
)

print(
    "Initial shock:",
    name_map[largest_bank]
)

print(
    "Total failed:",
    jpm_directed["total_failed"]
)

print(
    "Secondary failures:",
    jpm_directed["secondary_failures"]
)

print(
    "Rounds:",
    jpm_directed["rounds"]
)

print(
    "Total cumulative loss:",
    jpm_directed["total_cumulative_loss"]
)

Initial shock: JPMORGAN CHASE BANK, NATIONAL ASSOCIATION
Total failed: 11
Secondary failures: 10
Rounds: 2
Total cumulative loss: 690895841.3168138


In [166]:
# --------------------------------------------------
# Systemic impact — directed sparse network
# --------------------------------------------------

directed_results = []

for bank in bank_ids:

    result = simulate_cascade(
        shocked_bank=bank,
        W=W_directed,
        bank_ids=bank_ids,
        tier1_map=t1_map,
        lgd=1.0
    )

    directed_results.append({
        "shocked_bank": bank,
        "total_failed": result["total_failed"],
        "secondary_failures": result["secondary_failures"],
        "rounds": result["rounds"],
        "total_cumulative_loss": (
            result["total_cumulative_loss"]
        ),
        "max_loss_to_tier1": (
            result["max_loss_to_tier1"]
        )
    })

directed_cascade_df = pd.DataFrame(
    directed_results
)

directed_cascade_df["Financial Institution Name"] = (
    directed_cascade_df["shocked_bank"]
    .map(name_map)
)

directed_cascade_df["is_mds"] = (
    directed_cascade_df["shocked_bank"]
    .isin(mds_set)
)

directed_cascade_df["total_assets"] = (
    directed_cascade_df["shocked_bank"]
    .map(asset_map)
)

directed_cascade_df["failure_probability"] = (
    directed_cascade_df["shocked_bank"]
    .map(
        risk_snapshot
        .set_index("IDRSSD")[
            "failure_probability"
        ]
    )
)

directed_in_map = dict(
    G_directed.in_degree()
)

directed_out_map = dict(
    G_directed.out_degree()
)

directed_cascade_df["contagion_in_degree"] = (
    directed_cascade_df["shocked_bank"]
    .map(directed_in_map)
)

directed_cascade_df["contagion_out_degree"] = (
    directed_cascade_df["shocked_bank"]
    .map(directed_out_map)
)

directed_cascade_df = (
    directed_cascade_df
    .sort_values(
        [
            "total_failed",
            "total_cumulative_loss"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

directed_cascade_df.head(20)

,shocked_bank,total_failed,secondary_failures,rounds,total_cumulative_loss,max_loss_to_tier1,Financial Institution Name,is_mds,total_assets,failure_probability,contagion_in_degree,contagion_out_degree
0,852218,11,10,2,6.908958e+08,2.038481,"JPMORGAN CHASE BANK, NATIONAL ASSOCIATION",True,3.788551e+09,0.000971,186,20
1,480228,6,5,2,5.555087e+08,4.626755,"BANK OF AMERICA, NATIONAL ASSOCIATION",False,2.665555e+09,0.006026,150,15
2,476810,4,3,2,2.417419e+08,4.601251,"CITIBANK, N.A.",True,1.833933e+09,0.000933,107,12
3,504713,3,2,2,1.118240e+08,2.068760,U.S. BANK NATIONAL ASSOCIATION,True,6.714381e+08,0.005403,58,7
4,451965,2,1,2,2.587432e+08,2.200968,"WELLS FARGO BANK, NATIONAL ASSOCIATION",True,1.746394e+09,0.000148,108,11
5,112837,2,1,2,9.683687e+07,1.999347,"CAPITAL ONE, NATIONAL ASSOCIATION",False,6.489094e+08,0.000653,58,7
6,852320,2,1,2,3.858282e+07,1.343757,TRUIST BANK,False,5.355850e+08,0.000159,43,7
7,2182786,1,0,1,4.893272e+07,0.654010,GOLDMAN SACHS BANK USA,True,6.254100e+08,0.000553,52,7
8,497404,1,0,1,4.178315e+07,0.618706,"TD BANK, NATIONAL ASSOCIATION",False,3.564471e+08,0.000069,28,6
9,817824,1,0,1,2.554108e+07,0.962249,"PNC BANK, NATIONAL ASSOCIATION",False,5.545731e+08,0.000291,36,7


In [167]:
# --------------------------------------------------
# Separate direct-loss and cascade-impact measures
# --------------------------------------------------

directed_cascade_df["system_loss_fraction"] = (
    directed_cascade_df["total_cumulative_loss"]
    / assets.sum()
)

directed_cascade_df["cascade_fraction"] = (
    directed_cascade_df["secondary_failures"]
    / (len(bank_ids) - 1)
)

directed_cascade_df[
    [
        "Financial Institution Name",
        "failure_probability",
        "total_cumulative_loss",
        "system_loss_fraction",
        "secondary_failures",
        "cascade_fraction",
        "contagion_in_degree",
        "contagion_out_degree",
        "is_mds"
    ]
].head(20)

,Financial Institution Name,failure_probability,total_cumulative_loss,system_loss_fraction,secondary_failures,cascade_fraction,contagion_in_degree,contagion_out_degree,is_mds
0,"JPMORGAN CHASE BANK, NATIONAL ASSOCIATION",0.000971,6.908958e+08,0.030833,10,0.033445,186,20,True
1,"BANK OF AMERICA, NATIONAL ASSOCIATION",0.006026,5.555087e+08,0.024791,5,0.016722,150,15,False
2,"CITIBANK, N.A.",0.000933,2.417419e+08,0.010788,3,0.010033,107,12,True
3,U.S. BANK NATIONAL ASSOCIATION,0.005403,1.118240e+08,0.004990,2,0.006689,58,7,True
4,"WELLS FARGO BANK, NATIONAL ASSOCIATION",0.000148,2.587432e+08,0.011547,1,0.003344,108,11,True
5,"CAPITAL ONE, NATIONAL ASSOCIATION",0.000653,9.683687e+07,0.004322,1,0.003344,58,7,False
6,TRUIST BANK,0.000159,3.858282e+07,0.001722,1,0.003344,43,7,False
7,GOLDMAN SACHS BANK USA,0.000553,4.893272e+07,0.002184,0,0.000000,52,7,True
8,"TD BANK, NATIONAL ASSOCIATION",0.000069,4.178315e+07,0.001865,0,0.000000,28,6,False
9,"PNC BANK, NATIONAL ASSOCIATION",0.000291,2.554108e+07,0.001140,0,0.000000,36,7,False


In [168]:
# --------------------------------------------------
# Robustness across synthetic-network random seeds
# --------------------------------------------------

def build_directed_network(seed):
    """
    Build one sparse directed exposure matrix.

    Row sums exactly match observed aggregate
    interbank assets.
    """

    rng = np.random.default_rng(seed)

    W_new = np.zeros(
        (N, N),
        dtype=float
    )

    borrower_weight = assets / assets.sum()

    for i, lender in enumerate(bank_ids):

        target_exposure = ib_map[lender]

        lender_ratio = (
            assets[i] / assets.max()
        )

        k = int(
            MIN_COUNTERPARTIES
            + lender_ratio *
            (MAX_COUNTERPARTIES - MIN_COUNTERPARTIES)
        )

        k = max(
            MIN_COUNTERPARTIES,
            min(MAX_COUNTERPARTIES, k)
        )

        candidates = np.delete(
            np.arange(N),
            i
        )

        weights = borrower_weight[candidates]
        weights = weights / weights.sum()

        chosen = rng.choice(
            candidates,
            size=k,
            replace=False,
            p=weights
        )

        chosen_weights = borrower_weight[chosen]
        chosen_weights = (
            chosen_weights /
            chosen_weights.sum()
        )

        W_new[i, chosen] = (
            target_exposure *
            chosen_weights
        )

    return W_new


seeds = [42, 43, 44, 45, 46]

robustness_results = []

for seed in seeds:

    print(f"Running seed {seed}...")

    W_seed = build_directed_network(seed)

    # Verify aggregate exposure preservation
    row_error = np.abs(
        W_seed.sum(axis=1)
        - directed_observed
    ).max()

    # Run all shocks
    for bank in bank_ids:

        result = simulate_cascade(
            shocked_bank=bank,
            W=W_seed,
            bank_ids=bank_ids,
            tier1_map=t1_map,
            lgd=1.0
        )

        robustness_results.append({
            "seed": seed,
            "shocked_bank": bank,
            "total_failed": result["total_failed"],
            "secondary_failures": result[
                "secondary_failures"
            ],
            "rounds": result["rounds"],
            "total_cumulative_loss": result[
                "total_cumulative_loss"
            ],
            "max_loss_to_tier1": result[
                "max_loss_to_tier1"
            ]
        })

    print(
        "Max row-sum error:",
        row_error
    )


robustness_df = pd.DataFrame(
    robustness_results
)

print("\nCompleted.")
print(
    "Rows:",
    len(robustness_df)
)

Running seed 42...
Max row-sum error: 5.960464477539063e-08
Running seed 43...
Max row-sum error: 2.9802322387695312e-08
Running seed 44...
Max row-sum error: 5.960464477539063e-08
Running seed 45...
Max row-sum error: 3.725290298461914e-09
Running seed 46...
Max row-sum error: 5.960464477539063e-08

Completed.
Rows: 1500


In [169]:
# --------------------------------------------------
# JPMorgan robustness
# --------------------------------------------------

jpm_robust = robustness_df[
    robustness_df["shocked_bank"] == largest_bank
].copy()

jpm_robust[
    [
        "seed",
        "total_failed",
        "secondary_failures",
        "rounds",
        "total_cumulative_loss",
        "max_loss_to_tier1"
    ]
]

,seed,total_failed,secondary_failures,rounds,total_cumulative_loss,max_loss_to_tier1
266,42,12,11,2,5.547981e+08,4.495622
566,43,18,17,3,7.953555e+08,3.502374
866,44,15,14,2,6.440732e+08,3.172313
1166,45,12,11,2,6.296118e+08,4.636726
1466,46,11,10,2,6.824625e+08,3.919881


In [170]:
# Summary statistics for JPMorgan

jpm_robust[
    [
        "secondary_failures",
        "total_cumulative_loss",
        "rounds"
    ]
].describe()

,secondary_failures,total_cumulative_loss,rounds
count,5.000000,5.000000e+00,5.000000
mean,12.600000,6.612602e+08,2.200000
std,2.880972,8.812249e+07,0.447214
min,10.000000,5.547981e+08,2.000000
25%,11.000000,6.296118e+08,2.000000
50%,11.000000,6.440732e+08,2.000000
75%,14.000000,6.824625e+08,2.000000
max,17.000000,7.953555e+08,3.000000


In [171]:
# --------------------------------------------------
# Average systemic impact across network seeds
# --------------------------------------------------

seed_summary = (
    robustness_df
    .groupby("shocked_bank")
    .agg(
        mean_secondary_failures=(
            "secondary_failures",
            "mean"
        ),
        std_secondary_failures=(
            "secondary_failures",
            "std"
        ),
        mean_cumulative_loss=(
            "total_cumulative_loss",
            "mean"
        ),
        std_cumulative_loss=(
            "total_cumulative_loss",
            "std"
        ),
        mean_total_failed=(
            "total_failed",
            "mean"
        ),
        max_secondary_failures=(
            "secondary_failures",
            "max"
        )
    )
    .reset_index()
)

seed_summary["Financial Institution Name"] = (
    seed_summary["shocked_bank"].map(name_map)
)

seed_summary = seed_summary.sort_values(
    [
        "mean_secondary_failures",
        "mean_cumulative_loss"
    ],
    ascending=False
).reset_index(drop=True)

seed_summary.head(20)

,shocked_bank,mean_secondary_failures,std_secondary_failures,mean_cumulative_loss,std_cumulative_loss,mean_total_failed,max_secondary_failures,Financial Institution Name
0,852218,12.6,2.880972,6.612602e+08,8.812249e+07,13.6,17,"JPMORGAN CHASE BANK, NATIONAL ASSOCIATION"
1,480228,6.6,0.547723,6.172843e+08,4.629604e+07,7.6,7,"BANK OF AMERICA, NATIONAL ASSOCIATION"
2,476810,2.4,2.190890,2.914539e+08,7.197572e+07,3.4,4,"CITIBANK, N.A."
3,451965,2.4,1.816590,2.909251e+08,2.969702e+07,3.4,5,"WELLS FARGO BANK, NATIONAL ASSOCIATION"
4,112837,0.2,0.447214,8.694406e+07,3.661944e+07,1.2,1,"CAPITAL ONE, NATIONAL ASSOCIATION"
5,2182786,0.2,0.447214,7.100670e+07,2.229496e+06,1.2,1,GOLDMAN SACHS BANK USA
6,504713,0.0,0.000000,5.922329e+07,2.661968e+07,1.0,0,U.S. BANK NATIONAL ASSOCIATION
7,852320,0.0,0.000000,4.339325e+07,1.434876e+07,1.0,0,TRUIST BANK
8,817824,0.0,0.000000,4.077810e+07,6.468359e+06,1.0,0,"PNC BANK, NATIONAL ASSOCIATION"
9,497404,0.0,0.000000,2.925432e+07,1.162649e+07,1.0,0,"TD BANK, NATIONAL ASSOCIATION"


In [172]:
# --------------------------------------------------
# Top-10 systemic-impact stability
# --------------------------------------------------

top10_by_seed = {}

for seed in seeds:

    seed_data = robustness_df[
        robustness_df["seed"] == seed
    ]

    top10 = set(
        seed_data
        .nlargest(
            10,
            "total_cumulative_loss"
        )["shocked_bank"]
    )

    top10_by_seed[seed] = top10

for seed in seeds:
    print(
        f"\nSeed {seed}:"
    )

    for bank in list(
        top10_by_seed[seed]
    )[:10]:
        print(
            " ",
            name_map.get(bank)
        )


Seed 42:
  TRUIST BANK
  CITIBANK, N.A.
  U.S. BANK NATIONAL ASSOCIATION
  GOLDMAN SACHS BANK USA
  PNC BANK, NATIONAL ASSOCIATION
  CAPITAL ONE, NATIONAL ASSOCIATION
  JPMORGAN CHASE BANK, NATIONAL ASSOCIATION
  STATE STREET BANK AND TRUST COMPANY
  BANK OF AMERICA, NATIONAL ASSOCIATION
  WELLS FARGO BANK, NATIONAL ASSOCIATION

Seed 43:
  TRUIST BANK
  CITIBANK, N.A.
  U.S. BANK NATIONAL ASSOCIATION
  GOLDMAN SACHS BANK USA
  PNC BANK, NATIONAL ASSOCIATION
  CAPITAL ONE, NATIONAL ASSOCIATION
  TD BANK, NATIONAL ASSOCIATION
  JPMORGAN CHASE BANK, NATIONAL ASSOCIATION
  BANK OF AMERICA, NATIONAL ASSOCIATION
  WELLS FARGO BANK, NATIONAL ASSOCIATION

Seed 44:
  CITIBANK, N.A.
  GOLDMAN SACHS BANK USA
  U.S. BANK NATIONAL ASSOCIATION
  PNC BANK, NATIONAL ASSOCIATION
  CAPITAL ONE, NATIONAL ASSOCIATION
  TD BANK, NATIONAL ASSOCIATION
  JPMORGAN CHASE BANK, NATIONAL ASSOCIATION
  BANK OF AMERICA, NATIONAL ASSOCIATION
  STATE STREET BANK AND TRUST COMPANY
  WELLS FARGO BANK, NATIONAL ASSOCIA

In [173]:
# --------------------------------------------------
# Pairwise top-10 overlap
# --------------------------------------------------

overlaps = []

for i in range(len(seeds)):

    for j in range(i + 1, len(seeds)):

        s1 = top10_by_seed[seeds[i]]
        s2 = top10_by_seed[seeds[j]]

        overlap = len(s1.intersection(s2))

        overlaps.append({
            "seed_1": seeds[i],
            "seed_2": seeds[j],
            "top10_overlap": overlap
        })

overlap_df = pd.DataFrame(overlaps)

print(overlap_df)

print(
    "\nMean top-10 overlap:",
    overlap_df["top10_overlap"].mean()
)

   seed_1  seed_2  top10_overlap
0      42      43              9
1      42      44              9
2      42      45             10
3      42      46              9
4      43      44              9
5      43      45              9
6      43      46              9
7      44      45              9
8      44      46              8
9      45      46              9

Mean top-10 overlap: 9.0


In [174]:
# --------------------------------------------------
# Borrower-size sensitivity
# --------------------------------------------------

def build_directed_network_alpha(
    seed,
    alpha
):
    """
    Build sparse directed exposure network.

    Borrower selection probability is proportional to:

        Assets^alpha

    alpha = 1.0  -> strong size preference
    alpha = 0.5  -> moderate size preference
    alpha = 0.25 -> weak size preference
    """

    rng = np.random.default_rng(seed)

    W_new = np.zeros(
        (N, N),
        dtype=float
    )

    # Size-dependent borrower attractiveness
    borrower_weight = assets.astype(float) ** alpha
    borrower_weight = (
        borrower_weight /
        borrower_weight.sum()
    )

    for i, lender in enumerate(bank_ids):

        target_exposure = ib_map[lender]

        # Same counterparties-per-lender rule
        lender_ratio = (
            assets[i] / assets.max()
        )

        k = int(
            MIN_COUNTERPARTIES
            + lender_ratio *
            (MAX_COUNTERPARTIES - MIN_COUNTERPARTIES)
        )

        k = max(
            MIN_COUNTERPARTIES,
            min(MAX_COUNTERPARTIES, k)
        )

        candidates = np.delete(
            np.arange(N),
            i
        )

        weights = borrower_weight[candidates]
        weights = (
            weights /
            weights.sum()
        )

        chosen = rng.choice(
            candidates,
            size=k,
            replace=False,
            p=weights
        )

        chosen_weights = borrower_weight[chosen]
        chosen_weights = (
            chosen_weights /
            chosen_weights.sum()
        )

        W_new[i, chosen] = (
            target_exposure *
            chosen_weights
        )

    return W_new

In [175]:
# --------------------------------------------------
# Compare different borrower-size assumptions
# --------------------------------------------------

alphas = [1.00, 0.50, 0.25]
seeds_alpha = [42, 43, 44, 45, 46]

alpha_results = []

for alpha in alphas:

    print(f"\nRunning alpha = {alpha}")

    for seed in seeds_alpha:

        W_alpha = build_directed_network_alpha(
            seed=seed,
            alpha=alpha
        )

        # Check accounting preservation
        max_error = np.abs(
            W_alpha.sum(axis=1)
            - directed_observed
        ).max()

        for bank in bank_ids:

            result = simulate_cascade(
                shocked_bank=bank,
                W=W_alpha,
                bank_ids=bank_ids,
                tier1_map=t1_map,
                lgd=1.0
            )

            alpha_results.append({
                "alpha": alpha,
                "seed": seed,
                "shocked_bank": bank,
                "total_failed": result[
                    "total_failed"
                ],
                "secondary_failures": result[
                    "secondary_failures"
                ],
                "rounds": result[
                    "rounds"
                ],
                "total_cumulative_loss": result[
                    "total_cumulative_loss"
                ],
                "max_loss_to_tier1": result[
                    "max_loss_to_tier1"
                ],
                "max_row_error": max_error
            })

alpha_df = pd.DataFrame(alpha_results)

print("\nCompleted.")
print("Rows:", len(alpha_df))


Running alpha = 1.0

Running alpha = 0.5

Running alpha = 0.25

Completed.
Rows: 4500


In [176]:
# --------------------------------------------------
# Compare borrower-size assumptions
# --------------------------------------------------

alpha_summary = (
    alpha_df
    .groupby("alpha")
    .agg(
        mean_secondary_failures=(
            "secondary_failures",
            "mean"
        ),
        max_secondary_failures=(
            "secondary_failures",
            "max"
        ),
        mean_total_failed=(
            "total_failed",
            "mean"
        ),
        max_total_failed=(
            "total_failed",
            "max"
        ),
        mean_cumulative_loss=(
            "total_cumulative_loss",
            "mean"
        ),
        max_cumulative_loss=(
            "total_cumulative_loss",
            "max"
        )
    )
    .reset_index()
)

alpha_summary

,alpha,mean_secondary_failures,max_secondary_failures,mean_total_failed,max_total_failed,mean_cumulative_loss,max_cumulative_loss
0,0.25,0.054667,2,1.054667,3,8.061388e+06,1.958253e+08
1,0.50,0.066000,5,1.066000,6,8.601160e+06,5.334375e+08
2,1.00,0.081333,17,1.081333,18,8.033702e+06,7.953555e+08


In [177]:
# --------------------------------------------------
# JPMorgan across borrower-size assumptions
# --------------------------------------------------

jpm_alpha = alpha_df[
    alpha_df["shocked_bank"] == largest_bank
].copy()

jpm_alpha[
    [
        "alpha",
        "seed",
        "secondary_failures",
        "total_failed",
        "rounds",
        "total_cumulative_loss"
    ]
].sort_values(
    ["alpha", "seed"]
)

,alpha,seed,secondary_failures,total_failed,rounds,total_cumulative_loss
3266,0.25,42,0,1,1,7.423904e+07
3566,0.25,43,1,2,2,4.982995e+06
3866,0.25,44,1,2,2,9.116898e+07
4166,0.25,45,1,2,2,9.905982e+07
4466,0.25,46,2,3,2,1.958253e+08
1766,0.50,42,3,4,2,5.629425e+07
2066,0.50,43,2,3,2,8.868205e+07
2366,0.50,44,2,3,2,2.800302e+08
2666,0.50,45,2,3,2,4.274079e+07
2966,0.50,46,5,6,2,5.334375e+08


In [178]:
jpm_alpha.groupby("alpha").agg(
    mean_secondary_failures=(
        "secondary_failures",
        "mean"
    ),
    std_secondary_failures=(
        "secondary_failures",
        "std"
    ),
    mean_cumulative_loss=(
        "total_cumulative_loss",
        "mean"
    ),
    min_secondary_failures=(
        "secondary_failures",
        "min"
    ),
    max_secondary_failures=(
        "secondary_failures",
        "max"
    )
)

,mean_secondary_failures,std_secondary_failures,mean_cumulative_loss,min_secondary_failures,max_secondary_failures
alpha,,,,,
0.25,1.0,0.707107,9.305523e+07,0,2
0.50,2.8,1.303840,2.002370e+08,2,5
1.00,12.6,2.880972,6.612602e+08,10,17


In [179]:
# --------------------------------------------------
# How often does each bank appear in the top 10?
# --------------------------------------------------

top10_records = []

for alpha in alphas:

    for seed in seeds_alpha:

        subset = alpha_df[
            (alpha_df["alpha"] == alpha) &
            (alpha_df["seed"] == seed)
        ]

        top10 = subset.nlargest(
            10,
            "total_cumulative_loss"
        )

        for bank in top10["shocked_bank"]:
            top10_records.append({
                "alpha": alpha,
                "seed": seed,
                "shocked_bank": bank
            })

top10_frequency = pd.DataFrame(
    top10_records
)

frequency_summary = (
    top10_frequency
    .groupby("shocked_bank")
    .agg(
        appearances=("shocked_bank", "size")
    )
    .reset_index()
)

frequency_summary["Financial Institution Name"] = (
    frequency_summary["shocked_bank"]
    .map(name_map)
)

frequency_summary["appearance_rate"] = (
    frequency_summary["appearances"]
    / (len(alphas) * len(seeds_alpha))
)

frequency_summary = frequency_summary.sort_values(
    "appearances",
    ascending=False
)

frequency_summary.head(20)

,shocked_bank,appearances,Financial Institution Name,appearance_rate
23,480228,13,"BANK OF AMERICA, NATIONAL ASSOCIATION",0.866667
0,112837,12,"CAPITAL ONE, NATIONAL ASSOCIATION",0.800000
36,852218,12,"JPMORGAN CHASE BANK, NATIONAL ASSOCIATION",0.800000
21,451965,12,"WELLS FARGO BANK, NATIONAL ASSOCIATION",0.800000
22,476810,10,"CITIBANK, N.A.",0.666667
8,2182786,9,GOLDMAN SACHS BANK USA,0.600000
28,504713,9,U.S. BANK NATIONAL ASSOCIATION,0.600000
37,852320,8,TRUIST BANK,0.533333
35,817824,8,"PNC BANK, NATIONAL ASSOCIATION",0.533333
18,35301,6,STATE STREET BANK AND TRUST COMPANY,0.400000


In [180]:
# --------------------------------------------------
# Final robustness-averaged contagion impact
# --------------------------------------------------

contagion_summary = (
    alpha_df
    .groupby("shocked_bank")
    .agg(
        mean_secondary_failures=(
            "secondary_failures",
            "mean"
        ),
        std_secondary_failures=(
            "secondary_failures",
            "std"
        ),
        mean_total_failed=(
            "total_failed",
            "mean"
        ),
        mean_cumulative_loss=(
            "total_cumulative_loss",
            "mean"
        ),
        std_cumulative_loss=(
            "total_cumulative_loss",
            "std"
        ),
        max_secondary_failures=(
            "secondary_failures",
            "max"
        )
    )
    .reset_index()
)

contagion_summary["Financial Institution Name"] = (
    contagion_summary["shocked_bank"]
    .map(name_map)
)

contagion_summary = contagion_summary.sort_values(
    [
        "mean_secondary_failures",
        "mean_cumulative_loss"
    ],
    ascending=False
).reset_index(drop=True)

contagion_summary.head(20)

,shocked_bank,mean_secondary_failures,std_secondary_failures,mean_total_failed,mean_cumulative_loss,std_cumulative_loss,max_secondary_failures,Financial Institution Name
0,852218,5.466667,5.553206,6.466667,3.181841e+08,2.849326e+08,17,"JPMORGAN CHASE BANK, NATIONAL ASSOCIATION"
1,480228,3.133333,2.669047,4.133333,2.614411e+08,2.641970e+08,7,"BANK OF AMERICA, NATIONAL ASSOCIATION"
2,451965,1.600000,1.502379,2.600000,1.639055e+08,1.188314e+08,5,"WELLS FARGO BANK, NATIONAL ASSOCIATION"
3,476810,1.466667,1.597617,2.466667,1.689970e+08,1.287952e+08,4,"CITIBANK, N.A."
4,112837,0.533333,0.743223,1.533333,8.496642e+07,5.256852e+07,2,"CAPITAL ONE, NATIONAL ASSOCIATION"
5,2182786,0.266667,0.457738,1.266667,6.406168e+07,3.407320e+07,1,GOLDMAN SACHS BANK USA
6,852320,0.266667,0.593617,1.266667,5.076965e+07,4.110293e+07,2,TRUIST BANK
7,504713,0.266667,0.457738,1.266667,4.703390e+07,2.206877e+07,1,U.S. BANK NATIONAL ASSOCIATION
8,501105,0.266667,0.593617,1.266667,3.599312e+07,5.323255e+07,2,MANUFACTURERS AND TRADERS TRUST COMPANY
9,75633,0.266667,0.798809,1.266667,2.882747e+07,4.304161e+07,3,BMO BANK NATIONAL ASSOCIATION


In [181]:
# --------------------------------------------------
# Robust contagion score
# --------------------------------------------------

contagion_summary["cascade_percentile"] = (
    contagion_summary["mean_secondary_failures"]
    .rank(pct=True)
)

contagion_summary["loss_percentile"] = (
    contagion_summary["mean_cumulative_loss"]
    .rank(pct=True)
)

contagion_summary["contagion_score"] = (
    0.5 *
    contagion_summary["cascade_percentile"]
    +
    0.5 *
    contagion_summary["loss_percentile"]
)

contagion_summary = contagion_summary.sort_values(
    "contagion_score",
    ascending=False
).reset_index(drop=True)

contagion_summary.head(20)

,shocked_bank,mean_secondary_failures,std_secondary_failures,mean_total_failed,mean_cumulative_loss,std_cumulative_loss,max_secondary_failures,Financial Institution Name,cascade_percentile,loss_percentile,contagion_score
0,852218,5.466667,5.553206,6.466667,3.181841e+08,2.849326e+08,17,"JPMORGAN CHASE BANK, NATIONAL ASSOCIATION",1.000000,1.000000,1.000000
1,480228,3.133333,2.669047,4.133333,2.614411e+08,2.641970e+08,7,"BANK OF AMERICA, NATIONAL ASSOCIATION",0.996667,0.996667,0.996667
2,451965,1.600000,1.502379,2.600000,1.639055e+08,1.188314e+08,5,"WELLS FARGO BANK, NATIONAL ASSOCIATION",0.993333,0.990000,0.991667
3,476810,1.466667,1.597617,2.466667,1.689970e+08,1.287952e+08,4,"CITIBANK, N.A.",0.990000,0.993333,0.991667
4,112837,0.533333,0.743223,1.533333,8.496642e+07,5.256852e+07,2,"CAPITAL ONE, NATIONAL ASSOCIATION",0.986667,0.986667,0.986667
5,2182786,0.266667,0.457738,1.266667,6.406168e+07,3.407320e+07,1,GOLDMAN SACHS BANK USA,0.975000,0.983333,0.979167
6,852320,0.266667,0.593617,1.266667,5.076965e+07,4.110293e+07,2,TRUIST BANK,0.975000,0.980000,0.977500
7,504713,0.266667,0.457738,1.266667,4.703390e+07,2.206877e+07,1,U.S. BANK NATIONAL ASSOCIATION,0.975000,0.976667,0.975833
8,501105,0.266667,0.593617,1.266667,3.599312e+07,5.323255e+07,2,MANUFACTURERS AND TRADERS TRUST COMPANY,0.975000,0.966667,0.970833
9,75633,0.266667,0.798809,1.266667,2.882747e+07,4.304161e+07,3,BMO BANK NATIONAL ASSOCIATION,0.975000,0.960000,0.967500


In [182]:
# --------------------------------------------------
# Merge robust contagion with distress risk
# --------------------------------------------------

systemic_final = contagion_summary[
    [
        "shocked_bank",
        "Financial Institution Name",
        "mean_secondary_failures",
        "std_secondary_failures",
        "mean_total_failed",
        "mean_cumulative_loss",
        "std_cumulative_loss",
        "contagion_score"
    ]
].copy()

systemic_final = systemic_final.merge(
    risk_snapshot[
        [
            "IDRSSD",
            "failure_probability",
            "total_assets"
        ]
    ],
    left_on="shocked_bank",
    right_on="IDRSSD",
    how="left",
    validate="one_to_one"
)

systemic_final["distress_percentile"] = (
    systemic_final["failure_probability"]
    .rank(pct=True)
)

In [183]:
# --------------------------------------------------
# Add mT-MDS structure
# --------------------------------------------------

systemic_final["is_mds"] = (
    systemic_final["shocked_bank"]
    .isin(mds_set)
)

systemic_final["mtds_degree"] = (
    systemic_final["shocked_bank"]
    .map(degree_map)
)

systemic_final["network_coverage"] = (
    systemic_final["shocked_bank"]
    .map(coverage_count)
    .fillna(1)
    .astype(int)
)

In [184]:
# --------------------------------------------------
# Final systemic-risk screening score
# --------------------------------------------------

systemic_final["systemic_risk_score"] = (
    0.5 *
    systemic_final["distress_percentile"]
    +
    0.5 *
    systemic_final["contagion_score"]
)

systemic_final = systemic_final.sort_values(
    "systemic_risk_score",
    ascending=False
).reset_index(drop=True)

systemic_final[
    [
        "Financial Institution Name",
        "failure_probability",
        "mean_secondary_failures",
        "mean_cumulative_loss",
        "contagion_score",
        "mtds_degree",
        "is_mds",
        "systemic_risk_score"
    ]
].head(25)

,Financial Institution Name,failure_probability,mean_secondary_failures,mean_cumulative_loss,contagion_score,mtds_degree,is_mds,systemic_risk_score
0,"MORGAN STANLEY PRIVATE BANK, NATIONAL ASSOCIATION",0.772845,0.200000,2.282404e+07,0.951667,0,True,0.975833
1,"NORTHERN TRUST COMPANY, THE",0.487829,0.200000,2.132037e+07,0.946667,1,False,0.970000
2,STATE STREET BANK AND TRUST COMPANY,0.123504,0.133333,3.660432e+07,0.942500,0,True,0.949583
3,"BANK OF NEW YORK MELLON, THE",0.139585,0.133333,2.485969e+07,0.934167,0,True,0.948750
4,UBS BANK USA,0.375422,0.066667,1.115140e+07,0.851667,58,True,0.920833
5,BANCO POPULAR DE PUERTO RICO,0.068896,0.133333,1.180636e+07,0.905833,14,False,0.917917
6,"CHARLES SCHWAB BANK, SSB",0.043238,0.133333,1.941001e+07,0.920833,87,True,0.915417
7,EAST WEST BANK,0.037375,0.133333,2.277387e+07,0.930833,79,False,0.913750
8,"BANK OF AMERICA, NATIONAL ASSOCIATION",0.006026,3.133333,2.614411e+08,0.996667,2,False,0.880000
9,U.S. BANK NATIONAL ASSOCIATION,0.005403,0.266667,4.703390e+07,0.975833,32,True,0.859583


In [185]:
# --------------------------------------------------
# Robust final score components
# --------------------------------------------------

score_base = systemic_final.copy()

# Contagion percentile
score_base["contagion_percentile"] = (
    score_base["contagion_score"]
    .rank(pct=True)
)

# Distress percentile
score_base["distress_percentile"] = (
    score_base["failure_probability"]
    .rank(pct=True)
)

weights = [0.25, 0.50, 0.75]

weight_results = []

for w in weights:

    temp = score_base.copy()

    temp["score"] = (
        w * temp["distress_percentile"]
        +
        (1 - w) * temp["contagion_percentile"]
    )

    temp = temp.sort_values(
        "score",
        ascending=False
    ).reset_index(drop=True)

    temp["rank"] = (
        np.arange(len(temp)) + 1
    )

    temp["distress_weight"] = w

    weight_results.append(
        temp[
            [
                "shocked_bank",
                "Financial Institution Name",
                "score",
                "rank",
                "distress_weight"
            ]
        ]
    )

weight_df = pd.concat(
    weight_results,
    ignore_index=True
)

print("Weight sensitivity completed.")

Weight sensitivity completed.


In [186]:
# --------------------------------------------------
# Top 10 under each weighting
# --------------------------------------------------

for w in weights:

    print("\n" + "=" * 70)
    print(f"Distress weight = {w}")
    print("=" * 70)

    print(
        weight_df[
            weight_df["distress_weight"] == w
        ]
        .sort_values("rank")
        .head(10)
        [
            [
                "rank",
                "Financial Institution Name",
                "score"
            ]
        ]
        .to_string(index=False)
    )


Distress weight = 0.25
 rank                        Financial Institution Name    score
    1 MORGAN STANLEY PRIVATE BANK, NATIONAL ASSOCIATION 0.970000
    2                       NORTHERN TRUST COMPANY, THE 0.965833
    3               STATE STREET BANK AND TRUST COMPANY 0.951667
    4                      BANK OF NEW YORK MELLON, THE 0.945833
    5             BANK OF AMERICA, NATIONAL ASSOCIATION 0.938333
    6                                    EAST WEST BANK 0.921667
    7                    U.S. BANK NATIONAL ASSOCIATION 0.918333
    8                          CHARLES SCHWAB BANK, SSB 0.917500
    9                      BANCO POPULAR DE PUERTO RICO 0.912500
   10                                      UBS BANK USA 0.907500

Distress weight = 0.5
 rank                        Financial Institution Name    score
    1 MORGAN STANLEY PRIVATE BANK, NATIONAL ASSOCIATION 0.980000
    2                       NORTHERN TRUST COMPANY, THE 0.975000
    3               STATE STREET BANK AND T

In [187]:
# --------------------------------------------------
# Top-10 overlap across score weightings
# --------------------------------------------------

top10_scores = {}

for w in weights:

    top10_scores[w] = set(
        weight_df[
            weight_df["distress_weight"] == w
        ]
        .nsmallest(10, "rank")
        ["shocked_bank"]
    )

for i in range(len(weights)):

    for j in range(i + 1, len(weights)):

        w1 = weights[i]
        w2 = weights[j]

        overlap = len(
            top10_scores[w1]
            .intersection(top10_scores[w2])
        )

        print(
            f"Weight {w1} vs {w2}: "
            f"{overlap}/10 overlap"
        )

Weight 0.25 vs 0.5: 10/10 overlap
Weight 0.25 vs 0.75: 8/10 overlap
Weight 0.5 vs 0.75: 8/10 overlap


In [188]:
# --------------------------------------------------
# Baseline systemic-risk table
# --------------------------------------------------

final_systemic_table = score_base.copy()

final_systemic_table["systemic_risk_score"] = (
    0.5 *
    final_systemic_table["distress_percentile"]
    +
    0.5 *
    final_systemic_table["contagion_percentile"]
)

final_systemic_table = (
    final_systemic_table
    .sort_values(
        "systemic_risk_score",
        ascending=False
    )
    .reset_index(drop=True)
)

final_systemic_table = final_systemic_table[
    [
        "Financial Institution Name",
        "shocked_bank",
        "failure_probability",
        "mean_secondary_failures",
        "mean_cumulative_loss",
        "std_cumulative_loss",
        "contagion_score",
        "distress_percentile",
        "contagion_percentile",
        "mtds_degree",
        "is_mds",
        "systemic_risk_score"
    ]
]

final_systemic_table.head(25)

,Financial Institution Name,shocked_bank,failure_probability,mean_secondary_failures,mean_cumulative_loss,std_cumulative_loss,contagion_score,distress_percentile,contagion_percentile,mtds_degree,is_mds,systemic_risk_score
0,"MORGAN STANLEY PRIVATE BANK, NATIONAL ASSOCIATION",2489805,0.772845,0.200000,2.282404e+07,2.147098e+07,0.951667,1.000000,0.960000,0,True,0.980000
1,"NORTHERN TRUST COMPANY, THE",210434,0.487829,0.200000,2.132037e+07,2.721708e+07,0.946667,0.993333,0.956667,1,False,0.975000
2,STATE STREET BANK AND TRUST COMPANY,35301,0.123504,0.133333,3.660432e+07,2.711826e+07,0.942500,0.956667,0.950000,0,True,0.953333
3,"BANK OF NEW YORK MELLON, THE",541101,0.139585,0.133333,2.485969e+07,3.414027e+07,0.934167,0.963333,0.940000,0,True,0.951667
4,UBS BANK USA,3212149,0.375422,0.066667,1.115140e+07,1.022662e+07,0.851667,0.990000,0.880000,58,True,0.935000
5,BANCO POPULAR DE PUERTO RICO,940311,0.068896,0.133333,1.180636e+07,1.943062e+07,0.905833,0.930000,0.906667,14,False,0.918333
6,"CHARLES SCHWAB BANK, SSB",3150447,0.043238,0.133333,1.941001e+07,1.535338e+07,0.920833,0.910000,0.920000,87,True,0.915000
7,EAST WEST BANK,197478,0.037375,0.133333,2.277387e+07,4.369205e+07,0.930833,0.896667,0.930000,79,False,0.913333
8,"BANK OF AMERICA, NATIONAL ASSOCIATION",480228,0.006026,3.133333,2.614411e+08,2.641970e+08,0.996667,0.763333,0.996667,2,False,0.880000
9,U.S. BANK NATIONAL ASSOCIATION,504713,0.005403,0.266667,4.703390e+07,2.206877e+07,0.975833,0.743333,0.976667,32,True,0.860000


In [189]:
# --------------------------------------------------
# Baseline contagion impact
# alpha = 0.5, averaged across 5 seeds
# --------------------------------------------------

baseline_alpha = 0.50

baseline_contagion = (
    alpha_df[
        alpha_df["alpha"] == baseline_alpha
    ]
    .groupby("shocked_bank")
    .agg(
        mean_secondary_failures=(
            "secondary_failures",
            "mean"
        ),
        std_secondary_failures=(
            "secondary_failures",
            "std"
        ),
        mean_total_failed=(
            "total_failed",
            "mean"
        ),
        mean_cumulative_loss=(
            "total_cumulative_loss",
            "mean"
        ),
        std_cumulative_loss=(
            "total_cumulative_loss",
            "std"
        ),
        max_secondary_failures=(
            "secondary_failures",
            "max"
        )
    )
    .reset_index()
)

baseline_contagion["Financial Institution Name"] = (
    baseline_contagion["shocked_bank"]
    .map(name_map)
)

print(
    "Banks:",
    len(baseline_contagion)
)

baseline_contagion.head(20)

Banks: 300


,shocked_bank,mean_secondary_failures,std_secondary_failures,mean_total_failed,mean_cumulative_loss,std_cumulative_loss,max_secondary_failures,Financial Institution Name
0,1001152,0.0,0.000000,1.0,9.886767e+04,1.367178e+05,0,INTERNATIONAL BANK OF COMMERCE
1,1002878,0.0,0.000000,1.0,2.270450e+05,1.377783e+05,0,NORTHWEST BANK
2,100562,0.0,0.000000,1.0,1.267169e+06,2.067981e+06,0,TRI COUNTIES BANK
3,1011526,0.0,0.000000,1.0,1.762472e+05,1.256419e+05,0,CITY NATIONAL BANK OF WEST VIRGINIA
4,1017939,0.0,0.000000,1.0,1.559561e+06,2.655863e+06,0,UNITED COMMUNITY BANK
5,107244,0.0,0.000000,1.0,2.680488e+06,4.749621e+06,0,BANK OZK
6,112837,0.8,0.836660,1.8,1.142918e+08,6.318502e+07,2,"CAPITAL ONE, NATIONAL ASSOCIATION"
7,114260,0.0,0.000000,1.0,1.557815e+05,2.452309e+05,0,FIRST NATIONAL BANK ALASKA
8,1176881,0.0,0.000000,1.0,1.964905e+06,4.336591e+06,0,XD BANK
9,1187001,0.0,0.000000,1.0,1.578757e+06,2.401923e+06,0,INTERBANK


In [190]:
# --------------------------------------------------
# Baseline contagion score
# --------------------------------------------------

baseline_contagion["cascade_percentile"] = (
    baseline_contagion[
        "mean_secondary_failures"
    ].rank(pct=True)
)

baseline_contagion["loss_percentile"] = (
    baseline_contagion[
        "mean_cumulative_loss"
    ].rank(pct=True)
)

baseline_contagion["contagion_score"] = (
    0.5 *
    baseline_contagion["cascade_percentile"]
    +
    0.5 *
    baseline_contagion["loss_percentile"]
)

baseline_contagion = (
    baseline_contagion
    .sort_values(
        "contagion_score",
        ascending=False
    )
    .reset_index(drop=True)
)

baseline_contagion.head(20)

,shocked_bank,mean_secondary_failures,std_secondary_failures,mean_total_failed,mean_cumulative_loss,std_cumulative_loss,max_secondary_failures,Financial Institution Name,cascade_percentile,loss_percentile,contagion_score
0,852218,2.8,1.303840,3.8,2.002370e+08,2.093792e+08,5,"JPMORGAN CHASE BANK, NATIONAL ASSOCIATION",1.000000,1.000000,1.000000
1,451965,2.2,0.836660,3.2,1.620797e+08,8.731820e+07,3,"WELLS FARGO BANK, NATIONAL ASSOCIATION",0.996667,0.993333,0.995000
2,476810,1.6,1.140175,2.6,1.817699e+08,9.758551e+07,3,"CITIBANK, N.A.",0.990000,0.996667,0.993333
3,480228,2.0,0.707107,3.0,1.116308e+08,5.024601e+07,3,"BANK OF AMERICA, NATIONAL ASSOCIATION",0.993333,0.986667,0.990000
4,112837,0.8,0.836660,1.8,1.142918e+08,6.318502e+07,2,"CAPITAL ONE, NATIONAL ASSOCIATION",0.985000,0.990000,0.987500
5,852320,0.8,0.836660,1.8,7.643340e+07,6.031681e+07,2,TRUIST BANK,0.985000,0.980000,0.982500
6,501105,0.6,0.894427,1.6,6.564354e+07,8.338250e+07,2,MANUFACTURERS AND TRADERS TRUST COMPANY,0.971667,0.976667,0.974167
7,817824,0.6,0.547723,1.6,6.373403e+07,6.010840e+07,1,"PNC BANK, NATIONAL ASSOCIATION",0.971667,0.970000,0.970833
8,75633,0.6,1.341641,1.6,5.649347e+07,6.323534e+07,3,BMO BANK NATIONAL ASSOCIATION,0.971667,0.966667,0.969167
9,2182786,0.4,0.547723,1.4,9.447307e+07,3.061313e+07,1,GOLDMAN SACHS BANK USA,0.948333,0.983333,0.965833


In [191]:
# --------------------------------------------------
# Final baseline systemic-risk table
# --------------------------------------------------

final_systemic = baseline_contagion[
    [
        "shocked_bank",
        "Financial Institution Name",
        "mean_secondary_failures",
        "std_secondary_failures",
        "mean_total_failed",
        "mean_cumulative_loss",
        "std_cumulative_loss",
        "contagion_score"
    ]
].copy()

# Attach 2025 distress prediction
risk_lookup = (
    risk_snapshot
    .set_index("IDRSSD")
)

final_systemic["failure_probability"] = (
    final_systemic["shocked_bank"]
    .map(
        risk_lookup["failure_probability"]
    )
)

final_systemic["total_assets"] = (
    final_systemic["shocked_bank"]
    .map(asset_map)
)

# Distress percentile
final_systemic["distress_percentile"] = (
    final_systemic[
        "failure_probability"
    ].rank(pct=True)
)

# MDS information
final_systemic["is_mds"] = (
    final_systemic["shocked_bank"]
    .isin(mds_set)
)

final_systemic["mtds_degree"] = (
    final_systemic["shocked_bank"]
    .map(degree_map)
)

final_systemic["network_coverage"] = (
    final_systemic["shocked_bank"]
    .map(coverage_count)
    .fillna(1)
    .astype(int)
)

# Final 50/50 baseline score
final_systemic["systemic_risk_score"] = (
    0.5 *
    final_systemic["distress_percentile"]
    +
    0.5 *
    final_systemic["contagion_score"]
)

final_systemic = (
    final_systemic
    .sort_values(
        "systemic_risk_score",
        ascending=False
    )
    .reset_index(drop=True)
)

final_systemic.head(25)

,shocked_bank,Financial Institution Name,mean_secondary_failures,std_secondary_failures,mean_total_failed,mean_cumulative_loss,std_cumulative_loss,contagion_score,failure_probability,total_assets,distress_percentile,is_mds,mtds_degree,network_coverage,systemic_risk_score
0,210434,"NORTHERN TRUST COMPANY, THE",0.4,0.894427,1.4,3.618409e+07,3.617148e+07,0.945833,0.487829,1.713099e+08,0.993333,False,1,1,0.969583
1,541101,"BANK OF NEW YORK MELLON, THE",0.4,0.547723,1.4,5.182261e+07,5.044657e+07,0.954167,0.139585,3.982930e+08,0.963333,True,0,1,0.958750
2,35301,STATE STREET BANK AND TRUST COMPANY,0.2,0.447214,1.2,5.323244e+07,3.700897e+07,0.937500,0.123504,3.717840e+08,0.956667,True,0,1,0.947083
3,2489805,"MORGAN STANLEY PRIVATE BANK, NATIONAL ASSOCIATION",0.2,0.447214,1.2,1.225043e+07,6.360190e+06,0.889167,0.772845,2.350340e+08,1.000000,True,0,1,0.944583
4,3212149,UBS BANK USA,0.2,0.447214,1.2,1.342468e+07,8.616353e+06,0.895833,0.375422,1.142949e+08,0.990000,True,58,3,0.942917
5,197478,EAST WEST BANK,0.4,0.894427,1.4,4.137991e+07,7.229080e+07,0.952500,0.037375,7.762036e+07,0.896667,False,79,1,0.924583
6,940311,BANCO POPULAR DE PUERTO RICO,0.2,0.447214,1.2,1.686899e+07,2.129018e+07,0.902500,0.068896,6.077800e+07,0.930000,False,14,1,0.916250
7,3150447,"CHARLES SCHWAB BANK, SSB",0.2,0.447214,1.2,2.035821e+07,1.718212e+07,0.907500,0.043238,2.476640e+08,0.910000,True,87,5,0.908750
8,480228,"BANK OF AMERICA, NATIONAL ASSOCIATION",2.0,0.707107,3.0,1.116308e+08,5.024601e+07,0.990000,0.006026,2.665555e+09,0.763333,False,2,1,0.876667
9,504713,U.S. BANK NATIONAL ASSOCIATION,0.6,0.547723,1.6,3.839351e+07,2.429499e+07,0.960833,0.005403,6.714381e+08,0.743333,True,32,2,0.852083


In [192]:
# --------------------------------------------------
# High-distress + high-contagion candidates
# --------------------------------------------------

final_systemic["high_distress"] = (
    final_systemic["distress_percentile"] >= 0.90
)

final_systemic["high_contagion"] = (
    final_systemic["contagion_score"].rank(pct=True)
    >= 0.90
)

priority_candidates = (
    final_systemic[
        final_systemic["high_distress"] &
        final_systemic["high_contagion"]
    ]
    .sort_values(
        "systemic_risk_score",
        ascending=False
    )
)

print(
    "Priority candidates:",
    len(priority_candidates)
)

priority_candidates[
    [
        "Financial Institution Name",
        "failure_probability",
        "mean_secondary_failures",
        "mean_cumulative_loss",
        "contagion_score",
        "mtds_degree",
        "is_mds",
        "systemic_risk_score"
    ]
]

Priority candidates: 7


,Financial Institution Name,failure_probability,mean_secondary_failures,mean_cumulative_loss,contagion_score,mtds_degree,is_mds,systemic_risk_score
0,"NORTHERN TRUST COMPANY, THE",0.487829,0.4,3.618409e+07,0.945833,1,False,0.969583
1,"BANK OF NEW YORK MELLON, THE",0.139585,0.4,5.182261e+07,0.954167,0,True,0.958750
2,STATE STREET BANK AND TRUST COMPANY,0.123504,0.2,5.323244e+07,0.937500,0,True,0.947083
3,"MORGAN STANLEY PRIVATE BANK, NATIONAL ASSOCIATION",0.772845,0.2,1.225043e+07,0.889167,0,True,0.944583
4,UBS BANK USA,0.375422,0.2,1.342468e+07,0.895833,58,True,0.942917
6,BANCO POPULAR DE PUERTO RICO,0.068896,0.2,1.686899e+07,0.902500,14,False,0.916250
7,"CHARLES SCHWAB BANK, SSB",0.043238,0.2,2.035821e+07,0.907500,87,True,0.908750


In [193]:
# --------------------------------------------------
# Save final baseline results
# --------------------------------------------------

final_systemic.to_csv(
    PROCESSED_DIR /
    "final_systemic_risk_baseline.csv",
    index=False
)

baseline_contagion.to_csv(
    PROCESSED_DIR /
    "baseline_contagion_alpha_05.csv",
    index=False
)

print(
    "Saved final systemic-risk baseline."
)

Saved final systemic-risk baseline.
